# Buzzer Classification — Local End-to-End Collector

One notebook, top to bottom, TikTok, Instagram, X, Facebook and YouTube, on your own machine.

**What "end-to-end" means here, honestly:** YouTube is a pure API call — that cell runs with
zero intervention. The other four require a real login and real scrolling; collection uses the browser session and platform responses. Those cells **pause and
wait for you** — a Chromium window opens, you log in and/or scroll, you press Enter in the
notebook, execution continues automatically from there. Run the whole notebook with "Run All";
it will simply stop and wait for you exactly where a human is genuinely required, then proceed
on its own.

Login is a one-time cost per platform. Each browser profile persists to disk
(`browser_profile/<platform>/`), so the second time you collect from a given platform you likely
won't be prompted to log in again — only to confirm the session still looks valid.

## Setup (once)

```bash
pip install playwright google-api-python-client "pandas>=2.2,<3" numpy langdetect python-dotenv scikit-learn
playwright install chromium
```

Set one environment variable before launching Jupyter (or paste it into §1 — see the warning
there about not committing it):

```bash
export YOUTUBE_API_KEY="..."
```

## Edit before running

Section 2 is the only cell you need to change: video IDs and the post URLs (TikTok, Instagram,
X, Facebook) you want to collect comments from. Everything after that runs unattended except the
browser login pauses, and writes CSV files each time you run it.
**Current export:** all platforms write only `like_count`, `reply_count`, and `date_published` (comment publication time in UTC). Unknown values use `\N`. Raw payload archives remain local for deduplication/rebuilding. Profile enrichment and legacy account-feature exports are not run. Existing historical files are not deleted.


## 1. Environment

In [ ]:
import os, re, json, time, hashlib, asyncio, unicodedata
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
from collections import Counter
from typing import Any, Iterable

import pandas as pd
import numpy as np

COLLECTOR_VERSION = "local-e2e-1.1.0"

ROOT      = Path("./buzzer_data")
RAW       = ROOT / "raw"
CANONICAL = ROOT / "canonical"
FEATURES  = ROOT / "features"
PROFILE   = ROOT / "browser_profile"
for p in (RAW, CANONICAL, FEATURES, PROFILE):
    p.mkdir(parents=True, exist_ok=True)

from dotenv import load_dotenv
load_dotenv(override=False)
YOUTUBE_API_KEY = os.environ.get("YOUTUBE_API_KEY")

def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat()

def jsonl_append(path: Path, rows: Iterable[dict]) -> int:
    n = 0
    with path.open("a", encoding="utf-8") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False, default=str) + "\n"); n += 1
    return n

def jsonl_read(path: Path) -> list[dict]:
    if not path.exists(): return []
    with path.open(encoding="utf-8") as fh:
        return [json.loads(l) for l in fh if l.strip()]

print("collector", COLLECTOR_VERSION)
print("YouTube key:", "set" if YOUTUBE_API_KEY else "not set; YouTube will be skipped")
def jsonl_iter(path: Path):
    """Stream payloads; report corrupt records instead of silently dropping observations."""
    if not path.exists():
        return
    with path.open(encoding="utf-8") as fh:
        for line_no, line in enumerate(fh, 1):
            if line.strip():
                try:
                    yield json.loads(line)
                except json.JSONDecodeError as exc:
                    raise ValueError(f"{path}:{line_no}: invalid JSON") from exc


## 2. What to collect — edit this cell

`BROWSER_TARGETS` entries need a `platform` (`tiktok`, `instagram`, `x`, or `facebook`) and the
direct URL to the post whose comments you want. `max_comments` is the primary stop condition per
post; `batch_rounds`/`max_batches`/`idle_limit`/`pause_ms` tune how the scroll-and-check loop
behaves (see the comments above `BROWSER_TARGETS` below).

In [ ]:
#@ ---- YouTube ----
YT_VIDEO_IDS         = ["gksiY4Zxb54"]                             # e.g. ["dQw4w9WgXcQ"]
YT_MAX_PER_VIDEO     = 500     # capped per video — breadth (more videos) beats depth (one huge one)

#@ ---- TikTok / Instagram / X / Facebook — one shared Chromium bot, see §5 ----
# Strategy: many same-topic posts, each capped, rather than one post scraped to exhaustion —
# coordination features only fire when the same accounts recur across posts, so breadth matters
# more than draining any single post's comment section.
# max_comments: PRIMARY stop — cap unique comments captured per post (~300-800 is a good range).
# target_coverage: legacy setting retained for compatibility; totals are unverified hints.
# Stop conditions are the cap, unique-comment plateau, or batch ceiling.
# batch_rounds/max_batches: scrolls in batches of batch_rounds, up to max_batches batches,
#   checking real coverage after each. Total ceiling is roughly max_batches * (batch_rounds + ~30).
# pause_ms is a ceiling, not a fixed wait — each round polls for new content and moves on as
#   soon as it arrives, so most rounds finish well under this.
#
# Each platform gets its own persistent login (browser_profile/<platform>/) and its own
# one-time login pause the first time you collect from it — see §5. Instagram, X and Facebook
# entries are commented out below; uncomment and point at a real post to enable them.
# Original target plus 12 research candidates; these are NOT confirmed buzzer labels.
# Candidate links come from indexed metadata; live availability must be checked during collection.
TIKTOK_VIDEO_URLS = [
    "https://www.tiktok.com/@yehezsilva/video/7673383392859245831",
    "https://www.tiktok.com/@yanbo25/video/7538780346041552133",
    "https://www.tiktok.com/@ilhamber.2/video/7539883931223559432",
    "https://www.tiktok.com/@anies.hub/video/7416291932562853125",
    "https://www.tiktok.com/@muhammad.ilhamsyah23/video/7538464891959627024",
    "https://www.tiktok.com/@hamdiaadi/video/7673309762053770514",
    "https://www.tiktok.com/@garudasakti95/video/7544041220180430098",
    "https://www.tiktok.com/@memepolitaik1/video/7680473381287841045",
    "https://www.tiktok.com/@anomali.klipers/video/7675741914930433300",
    "https://www.tiktok.com/@lawliett41/video/7650964123953515792",
    "https://www.tiktok.com/@studio.musyafa/video/7650504991370693908",
    "https://www.tiktok.com/@cnnindonesia/video/7569811436185128213",
]

BROWSER_TARGETS = [
    *[{"platform": "tiktok",
       "url": url,
     "max_comments": 500, "target_coverage": 0.95, "batch_rounds": 60, "max_batches": 8,
     "idle_limit": 12, "pause_ms": 1800, "expand_replies": True}
      for url in TIKTOK_VIDEO_URLS],

    # {"platform": "instagram",
    #  "url": "https://www.instagram.com/p/<shortcode>/",
    #  "max_comments": 500, "target_coverage": 0.95, "batch_rounds": 60, "max_batches": 8,
    #  "idle_limit": 12, "pause_ms": 1800, "expand_replies": True},

    # {"platform": "x",
    #  "url": "https://x.com/<handle>/status/<id>",
    #  "max_comments": 500, "target_coverage": 0.95, "batch_rounds": 60, "max_batches": 8,
    #  # X interleaves ads/promoted content while scrolling, so plateaus can take a few extra
    #  # idle rounds to confirm — hence the higher idle_limit than the other platforms.
    #  "idle_limit": 16, "pause_ms": 1800, "expand_replies": True},

    # {"platform": "facebook",
    #  "url": "https://www.facebook.com/<page>/posts/<id>",
    #  "max_comments": 500, "target_coverage": 0.95, "batch_rounds": 60, "max_batches": 8,
    #  "idle_limit": 12, "pause_ms": 1800, "expand_replies": True},
]

print(f"YouTube: {len(YT_VIDEO_IDS)} videos")
print(f"Browser: {len(BROWSER_TARGETS)} targets across "
      f"{sorted(set(t['platform'] for t in BROWSER_TARGETS)) or 'none'}")
# Baseline event-time window [start, end), in UTC. None keeps the archive exploratory.
# Configure BOTH to export validated account-window JSONL; do not infer from account activity.
OBSERVATION_WINDOW_START = None
OBSERVATION_WINDOW_END = None
# Set only when confirmed by collector capability/diagnostics, never just from a null column.
# Example: {"facebook": {"median_comment_interval_seconds": "collection_failed"}}
PLATFORM_FEATURE_MISSINGNESS = {}


## 3. Canonical schema and text normalisation — shared by every platform

In [ ]:
PROFILE_AUDIT_FIELDS = ("profile_visibility", "profile_access_status", "profile_checked_at")

CANONICAL_FIELDS = [
    *PROFILE_AUDIT_FIELDS,
    "user_id", "username", "display_name", "account_created_at", "is_verified",
    "has_custom_avatar", "bio_text", "location",
    "followers_count", "following_count", "total_posts_count",
    "post_id", "source_post_id", "parent_comment_id", "thread_id", "created_at", "text_content", "clean_text", "language",
    "hashtags", "user_mentions", "urls", "media_types", "source_device",
    "like_count", "repost_count", "reply_count", "is_repost", "is_reply",
    "user_recent_posts", "user_recent_timestamps", "_text_observed",
    "_platform", "_profile_collected_at", "_profile_status", "_profile_user_id", "_collected_at", "_collector_version", "_source_url", "_raw_ref", "_capture_session", "_sampling_method", "_requested_comment_cap",
]
LIST_FIELDS = {"hashtags", "user_mentions", "urls", "media_types",
               "user_recent_posts", "user_recent_timestamps"}

def empty_record(**kw) -> dict:
    rec = {f: ([] if f in LIST_FIELDS else None) for f in CANONICAL_FIELDS}
    rec["profile_visibility"] = "unknown"
    rec["_collector_version"] = COLLECTOR_VERSION
    rec["_collected_at"] = now_utc()
    rec.update(kw)
    for field in IDENTITY_FIELDS:
        rec[field] = as_id(rec.get(field))
    if not rec.get("source_post_id"):
        rec["source_post_id"] = source_post_id(rec.get("_platform"), rec.get("_source_url"))
    return rec

def to_frame(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    for f in CANONICAL_FIELDS:
        if f not in df.columns:
            df[f] = [[] for _ in range(len(df))] if f in LIST_FIELDS else None
    return df[CANONICAL_FIELDS]

RE_URL     = re.compile(r"https?://\S+|www\.\S+", re.I)
RE_HASHTAG = re.compile("(?<!\\w)#([\\wÀ-ɏ؀-ۿ]+)", re.U)
RE_MENTION = re.compile(r"(?<!\w)@([\w.]+)", re.U)
RE_WS      = re.compile(r"\s+")
RE_ZWJ     = re.compile("[​-‏‪-‮⁠﻿]")
RE_RT      = re.compile(r"^RT @[\w]+:\s*")

def extract_entities(text: str) -> dict:
    t = text or ""
    return {"hashtags":      [h.lower() for h in RE_HASHTAG.findall(t)],
            "user_mentions": [m.lower().rstrip(".") for m in RE_MENTION.findall(t)],
            "urls":          RE_URL.findall(t)}

def clean_text(text: str) -> str:
    t = unicodedata.normalize("NFKC", text or "")
    t = RE_ZWJ.sub("", t); t = RE_RT.sub("", t)
    t = RE_URL.sub(" <url> ", t)
    t = RE_MENTION.sub(" <user> ", t)
    t = RE_HASHTAG.sub(lambda m: " " + m.group(1).lower() + " ", t)
    return RE_WS.sub(" ", t).strip()

def dedupe_key(text):
    # Preserve punctuation and emoji: "a+b" and "ab" must not become the same message.
    if not isinstance(text, str): return ""
    normalized = RE_WS.sub(" ", unicodedata.normalize("NFKC", text).casefold()).strip()
    return hashlib.sha256(normalized.encode()).hexdigest() if normalized else ""


try:
    from langdetect import detect, DetectorFactory, LangDetectException
    DetectorFactory.seed = 0
    def detect_language(t):
        t = (t or "").strip()
        if len(t) < 12: return None
        try: return detect(t)
        except LangDetectException: return None
except ImportError:
    def detect_language(t): return None

def enrich_text_fields(rec: dict) -> dict:
    rec["_text_observed"] = isinstance(rec.get("text_content"), str)
    txt = rec.get("text_content") or ""
    for field, values in extract_entities(txt).items():
        if not rec.get(field): rec[field] = values
    rec["clean_text"] = clean_text(txt)
    if not rec.get("language"): rec["language"] = detect_language(txt)
    return rec

def write_csv_with_lists(df: pd.DataFrame, path: Path) -> None:
    """CSV can't hold Python lists natively, so list-type columns are JSON-encoded on the way
    out. Always pair with read_csv_with_lists (or load_canonical) to get them back as lists
    rather than string reprs — otherwise every list-based feature silently reads as empty."""
    out = df.copy()
    for f in LIST_FIELDS:
        if f in out.columns:
            out[f] = out[f].map(lambda v: json.dumps(v if isinstance(v, list) else []))
    out.to_csv(path, index=False, na_rep="\\N")

def read_csv_with_lists(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        return None
    # Never infer IDs as numbers: 64-bit social IDs lose precision in float columns.
    header = pd.read_csv(path, nrows=0).columns
    df = pd.read_csv(path, dtype={c: "string" for c in header if c in IDENTITY_FIELDS
                                 or c in {"global_user_id", "username"}}, keep_default_na=False, na_values=["\\N"])
    # Version 1.1 writes an explicit null marker, preserving a known empty biography.
    legacy = df.get("_collector_version", pd.Series("", index=df.index)).ne("local-e2e-1.1.0")
    for field in df.columns:
        if field == "bio_text":
            df.loc[legacy & df[field].eq(""), field] = pd.NA
        elif field not in {"text_content", "clean_text"} and field not in LIST_FIELDS:
            df[field] = df[field].mask(df[field].eq(""))
    for f in LIST_FIELDS:
        if f in df.columns:
            df[f] = df[f].map(lambda v: json.loads(v) if isinstance(v, str) and v else [])
    return df

def scraping_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Export only requested comment metrics; unknown values remain null."""
    out = df.reindex(columns=["like_count", "reply_count"]).copy()
    dates = df["created_at"] if "created_at" in df else df.get("date_published", pd.Series(index=df.index, dtype="object"))
    out["date_published"] = pd.to_datetime(dates, utc=True, errors="coerce", format="mixed")
    return out


def save_canonical(df: pd.DataFrame, platform: str) -> Path:
    path = CANONICAL / f"{platform}.csv"
    write_csv_with_lists(scraping_columns(df), path)
    return path

def load_canonical(platform: str) -> pd.DataFrame | None:
    return read_csv_with_lists(CANONICAL / f"{platform}.csv")

print(len(CANONICAL_FIELDS), "canonical fields ready")
from urllib.parse import urlsplit, parse_qs

IDENTITY_FIELDS = {"user_id", "post_id", "source_post_id", "parent_comment_id", "thread_id", "_raw_ref", "_profile_user_id"}

def as_id(value):
    if value is None or pd.isna(value): return None
    value = str(value).strip()
    return value if value and value not in {"0", "nan", "None", "<NA>"} else None

def source_post_id(platform, url):
    if not isinstance(url, str) or not url: return None
    parsed = urlsplit(url)
    query = parse_qs(parsed.query)
    if platform == "youtube":
        if query.get("v"): return query["v"][0]
        if parsed.hostname in {"youtu.be", "www.youtu.be"}: return parsed.path.strip("/") or None
    patterns = {"tiktok": r"/video/([^/?]+)", "instagram": r"/(?:p|reel|reels)/([^/?]+)",
                "x": r"/status/([^/?]+)", "facebook": r"/(?:posts|videos|reel)/([^/?]+)"}
    match = re.search(patterns.get(platform, r"/(?:shorts|embed)/([^/?]+)"), parsed.path)
    if match: return match.group(1)
    if platform == "facebook":
        for key in ("story_fbid", "fbid", "v"):
            if query.get(key): return query[key][0]
    return None  # Unknown URL shapes stay missing, never collapse into a generic page URL.

def timestamp_utc(value):
    if value is None: return None
    try:
        if isinstance(value, (int, float)) or (isinstance(value, str) and value.isdigit()):
            return datetime.fromtimestamp(float(value), timezone.utc).isoformat()
        ts = pd.to_datetime(value, utc=True, errors="coerce")
        return None if pd.isna(ts) else ts.isoformat()
    except (ValueError, TypeError, OverflowError, OSError):
        return None

def nullable_bool(value):
    if value is None or pd.isna(value): return None
    if value in (True, 1, "1", "true", "True"): return True
    if value in (False, 0, "0", "false", "False"): return False
    return None

def profile_audit(user=None, readable=False, checked_at=None, privacy_key="privateAccount"):
    """Only an explicit privacy flag establishes private/public; failed reads are unknown.

    Restricted means explicitly private content, even if basic profile fields are readable.
    Null access/date means there is no recorded profile check.
    """
    user = user if isinstance(user, dict) else {}
    private = nullable_bool(user.get(privacy_key)) if readable else None
    return {
        "profile_visibility": "private" if private is True else "public" if private is False else "unknown",
        "profile_access_status": "restricted" if private is True else "readable" if readable else "failed",
        "profile_checked_at": checked_at or now_utc(),
    }


## 4. YouTube — runs unattended

In [ ]:
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

def yt_fetch(video_id, max_comments, yt=None):
    """Fetch top-level comments and paginated replies within one shared hard cap."""
    if max_comments < 0: raise ValueError("max_comments must be nonnegative")
    if max_comments == 0: return []
    yt = yt or build("youtube", "v3", developerKey=YOUTUBE_API_KEY, cache_discovery=False)
    out, seen, token, tokens = [], set(), None, set()
    def add(kind, item, parent=None):
        cid = item["snippet"]["topLevelComment"]["id"] if kind == "top" else item["id"]
        if cid not in seen and len(out) < max_comments:
            seen.add(cid)
            out.append({"_kind": kind, "_video_id": video_id, "_parent": parent,
                        "_captured_at": now_utc(), "_max_comments": max_comments, "item": item})
    while len(out) < max_comments:
        try:
            response = yt.commentThreads().list(part="snippet,replies", videoId=video_id,
                maxResults=min(100, max_comments-len(out)), pageToken=token,
                textFormat="plainText", order="time").execute()
        except HttpError as exc:
            raise RuntimeError(f"YouTube collection failed for {video_id}; canonical not replaced") from exc
        for item in response.get("items", []):
            add("top", item)
            parent = item["snippet"]["topLevelComment"]["id"]
            embedded = item.get("replies", {}).get("comments", [])
            for reply in embedded: add("reply", reply, parent)
            if len(embedded) < item["snippet"].get("totalReplyCount", 0):
                reply_token, reply_tokens = None, set()
                while len(out) < max_comments:
                    response_replies = yt.comments().list(part="snippet", parentId=parent,
                        maxResults=min(100, max_comments-len(out)), pageToken=reply_token,
                        textFormat="plainText").execute()
                    for reply in response_replies.get("items", []): add("reply", reply, parent)
                    reply_token = response_replies.get("nextPageToken")
                    if not reply_token or reply_token in reply_tokens: break
                    reply_tokens.add(reply_token)
            if len(out) >= max_comments: break
        token = response.get("nextPageToken")
        if not token or token in tokens: break
        tokens.add(token)
    return out

def yt_normalise(raw):
    top = raw["_kind"] == "top"
    item = raw["item"]
    comment = item["snippet"]["topLevelComment"] if top else item
    sn = comment["snippet"]
    parent = sn.get("parentId") or raw.get("_parent") if not top else None
    return enrich_text_fields(empty_record(
        _platform="youtube", _raw_ref=comment["id"],
        _collected_at=raw.get("_captured_at"), _sampling_method="time_order_threads_with_replies",
        _requested_comment_cap=raw.get("_max_comments"),
        _source_url=f"https://www.youtube.com/watch?v={raw['_video_id']}",
        source_post_id=raw["_video_id"], post_id=comment["id"],
        parent_comment_id=parent, thread_id=parent or comment["id"],
        user_id=(sn.get("authorChannelId") or {}).get("value"),
        display_name=sn.get("authorDisplayName"), created_at=sn.get("publishedAt"),
        text_content=sn.get("textOriginal") or sn.get("textDisplay") or "",
        like_count=sn.get("likeCount"),
        reply_count=item["snippet"].get("totalReplyCount") if top else None,
        is_reply=not top, is_repost=False, media_types=["text"]))

def yt_enrich_channels(channel_ids):
    yt, out = build("youtube","v3",developerKey=YOUTUBE_API_KEY,cache_discovery=False), {}
    ids = [c for c in dict.fromkeys(channel_ids) if c]
    for i in range(0, len(ids), 50):
        try:
            r = yt.channels().list(part="snippet,statistics", id=",".join(ids[i:i+50])).execute()
        except HttpError as e:
            print(f"  ! channels {i}: {e.reason}"); continue
        for ch in r.get("items", []):
            sn, st = ch.get("snippet",{}), ch.get("statistics",{})
            thumb = (sn.get("thumbnails",{}).get("default",{}) or {}).get("url","")
            out[ch["id"]] = {
                "username": (sn.get("customUrl") or "").lstrip("@") or None,
                "account_created_at": sn.get("publishedAt"),
                "bio_text": sn.get("description"), "location": sn.get("country"),
                "followers_count": int(st["subscriberCount"]) if not st.get("hiddenSubscriberCount")
                                   and "subscriberCount" in st else None,
                "total_posts_count": int(st["videoCount"]) if "videoCount" in st else None,
                "has_custom_avatar": None,
                "_profile_collected_at": now_utc(), "_profile_status": "observed",
            }
        time.sleep(0.1)
    return out

def run_youtube_collection():
    if not YOUTUBE_API_KEY:
        print("YOUTUBE_API_KEY not set — skipping YouTube"); return None
    if not YT_VIDEO_IDS:
        print("no YT_VIDEO_IDS configured — skipping YouTube"); return None
    recs = []
    for vid in YT_VIDEO_IDS:
        raw = yt_fetch(vid, YT_MAX_PER_VIDEO)
        jsonl_append(RAW / "youtube_raw.jsonl", raw)
        recs += [yt_normalise(r) for r in raw]
        print(f"  {vid}: {len(raw)} comments")
    if not recs:
        print("youtube: nothing collected"); return None
    df = to_frame(recs).drop_duplicates(subset=["post_id"], keep="last")
    save_canonical(df, "youtube")
    print(f"youtube: {len(df)} comments, {df.user_id.nunique()} accounts")
    return df

df_yt = run_youtube_collection()

## 5. Browser platforms — TikTok, Instagram, X, Facebook — the part that pauses for you

`Harvester` opens a real, visible Chromium window and records the JSON that the comment section
actually loads over the network, rather than parsing the DOM. It's the SAME class and the same
scroll/batch/coverage machinery for all four platforms below — only the endpoint patterns,
comment-panel selector, and reply-button text differ per platform (see the dicts at the top of
the next cell). Each platform uses its own persistent profile
(`browser_profile/<platform>/`), so a login you complete once is still there next time you run
this notebook.

**When you run the next cell:** for each platform present in `BROWSER_TARGETS`, a browser opens.
Log in if asked. Then come back to the notebook and press **Enter in the input box that appears
below the cell** — execution resumes automatically and scrolls the comment section for you. This
is the one genuinely manual step in the whole notebook, and it is manual because making it
automatic would mean automating a login, which is where "scraping public data" turns into
"circumventing platform security controls."

A few things differ meaningfully between platforms, worth knowing before you collect:
- **TikTok and Instagram** render a post at desktop width with a dedicated comments panel, so
  the harvester hovers that panel before scrolling.
- **X** has no such panel — replies scroll with the whole page/timeline — and can interleave
  ads/promoted tweets while scrolling, which is why its example config in §2 uses a higher
  `idle_limit`.
- **Facebook** is the least stable of the four: its GraphQL doc_ids rotate constantly and the
  comment payload shape has changed more than once. Expect to lean on `inspect_payloads` (§6)
  more here than on the other platforms if a run captures 0 or very few comments.

In [ ]:
# Sync Playwright API, driven from a dedicated worker thread.
#
# Two separate Windows/Jupyter issues stack here, and both need fixing:
#
# 1. ipykernel keeps an asyncio loop running in the main thread for its own use (zmq comms),
#    and Playwright's sync API refuses to run inside a thread that already has a loop
#    running ("Sync API inside the asyncio loop" error). Fix: call it from a fresh thread
#    that has no loop of its own — the ThreadPoolExecutor below.
# 2. That's not sufficient on Windows. ipykernel deliberately sets the *global* asyncio
#    event loop policy to WindowsSelectorEventLoopPolicy at startup, because pyzmq (which
#    the kernel's comms depend on) doesn't support ProactorEventLoop. That policy is
#    process-wide, not thread-local, so even a brand-new thread inherits it — and
#    Playwright's driver process needs Proactor to be spawned at all, hence the
#    NotImplementedError. Fix: explicitly switch the policy to Proactor before Playwright
#    starts. This is safe: it only changes what future new_event_loop() calls hand out —
#    the kernel's own loop object was already created at startup and keeps running as-is,
#    unaffected by a later policy change.
import sys, asyncio
if sys.platform.startswith("win"):
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

from playwright.sync_api import sync_playwright
from concurrent.futures import ThreadPoolExecutor

# One Harvester drives all four platforms — only the bits below differ per platform. Endpoint
# substrings are how each platform's comment payload gets recognised on the wire; everything
# else (scrolling, batching, flush/cache) is shared in the class further down.
COMMENT_ENDPOINTS = {
    "tiktok":    ["/api/comment/list", "/api/comment/reply"],
    # IG's classic private-API path. IG has intermittently routed comment loading through
    # GraphQL (`/graphql/query` + a doc_id) instead on some rollouts — if a run captures 0
    # payloads, check the Network tab on a real post and add whatever URL actually fires.
    "instagram": ["/api/v1/media/", "/comments/"],
    # X's GraphQL operation name is baked into the URL path itself
    # (".../i/api/graphql/<hash>/TweetDetail"), so matching the operation name survives the
    # hash rotating on every X deploy, which matching the hash itself would not.
    "x":         ["TweetDetail", "ConversationTimeline"],
    # Facebook's GraphQL doc_ids rotate constantly and carry no stable name in the URL, so
    # this is the widest (least precise) pattern here — parse_facebook sifts payloads by
    # shape rather than trusting the URL alone. Expect this one to need the most hand-tuning.
    "facebook":  ["/api/graphql/"],
}

# Endpoints that carry PROFILE data (followers/following/bio/verified) rather than comments,
# for platforms where visiting a profile page triggers a clean network response for it.
# TikTok isn't here — its profile data is server-rendered into a DOM script tag instead (see
# _extract_profile_from_page in §6), not fetched as a separate XHR. X isn't here either — its
# TweetDetail response already embeds the commenter's profile stats inline (see parse_x), so no
# separate profile visit is needed for X at all. Facebook profile enrichment isn't implemented
# (see the note in §6) so it has no entry.
PROFILE_ENDPOINTS = {
    "instagram": ["/api/v1/users/web_profile_info/"],
}

# Selector guesses for each platform's scrollable comment container, tried in order. TikTok and
# Instagram render a single post at desktop width as media-left / comments-right, so hovering
# the right-hand panel before wheeling is what makes page.mouse.wheel() land on the comment list
# instead of the whole document. X has no such panel — replies scroll with the main timeline —
# and Facebook's DOM is too unstable to target reliably, so both just use the viewport fallback.
COMMENT_PANEL_SELECTORS = {
    "tiktok":    ['[data-e2e="comment-list"]'],
    "instagram": ['section main ul', 'article ul'],
}
PANEL_FALLBACK = {  # (x_fraction, y_fraction) of the viewport, used when no selector matches
    "tiktok": (0.75, 0.5), "instagram": (0.75, 0.5), "x": (0.5, 0.5), "facebook": (0.5, 0.5),
}

# "view replies" button text varies per platform (and locale — id-ID strings included since
# the persistent context below runs in id-ID).
REPLY_BUTTON_PATTERNS = {
    "tiktok":    r"repl(y|ies)|balasan",
    "instagram": r"view (\d+ )?repl(y|ies)|lihat balasan",
    "x":         r"show (more )?repl(y|ies)|more repl(y|ies)",
    "facebook":  r"view (more|\d+) (repl(y|ies)|comments?)|lihat balasan|balasan lainnya",
}

# Populated by run_browser_collection: platform -> unique comments captured THIS session
# (before raw JSONL accumulation from earlier runs). §6 uses this to explain any gap between
# "what this run scrolled" and "what's in the canonical file" (which includes older runs too).
SESSION_SEEN: dict[str, int] = {}

# Coverage counts are diagnostic candidates. Platform totals can count different populations;
# the collector stops on a cap/plateau/batch ceiling, never on an unverified percentage.
def _walk_for_ids_and_totals(node):
    """Count comment-shaped nodes only; profile/post engagement totals are not coverage."""
    stack = [node]
    while stack:
        item = stack.pop()
        if isinstance(item, dict):
            cid = None
            if "cid" in item and "text" in item: cid = item["cid"]
            elif "text" in item and isinstance(item.get("user"), dict):
                cid = item.get("pk") or item.get("id")
            elif isinstance(item.get("legacy"), dict) and "full_text" in item["legacy"]:
                legacy = item["legacy"]
                if legacy.get("in_reply_to_status_id_str"):
                    cid = item.get("rest_id") or legacy.get("id_str")
                if cid is not None: yield "id", str(cid)
                continue  # quoted tweets and their users are not additional comments
            elif item.get("__typename") == "Comment" or "parent_comment" in item:
                cid = item.get("id") or item.get("legacy_fbid")
            if cid is not None: yield "id", str(cid)
            stack.extend(item.values())
        elif isinstance(item, list): stack.extend(item)

def coverage_parts(row):
    if row.get("_kind", "comment") != "comment": return set(), None
    ids = {val for kind, val in _walk_for_ids_and_totals(row.get("payload")) if kind == "id"}
    if row.get("_platform") == "x":
        target = source_post_id("x", row.get("_page_url"))
        ids = set()
        stack = [row.get("payload")]
        while stack:
            node = stack.pop()
            if isinstance(node, dict):
                legacy = node.get("legacy")
                if isinstance(legacy, dict) and "full_text" in legacy:
                    tid = as_id(node.get("rest_id") or legacy.get("id_str"))
                    if tid and tid != target and target and (legacy.get("conversation_id_str") == target
                            or legacy.get("in_reply_to_status_id_str") == target): ids.add(tid)
                    continue
                stack.extend(node.values())
            elif isinstance(node, list): stack.extend(node)
    payload = row.get("payload", {})
    total = None
    # Only top-level comment lists have a comparable denominator. Reply totals are per thread.
    url = row.get("_url", "")
    if isinstance(payload, dict) and "comments" in payload and "reply" not in url and "child" not in url:
        for key in ("total", "comment_count", "total_comment_count"):
            value = payload.get(key)
            if isinstance(value, int) and not isinstance(value, bool) and value >= 0:
                total = value; break
    return ids, total

def estimate_coverage(rows):
    by_target = {}
    for row in rows:
        key = (row.get("_platform"), source_post_id(row.get("_platform"), row.get("_page_url")))
        ids, total = coverage_parts(row)
        state = by_target.setdefault(key, [set(), None])
        state[0].update(ids)
        if total is not None: state[1] = max(state[1] or 0, total)
    seen = sum(len(state[0]) for state in by_target.values())
    totals = [state[1] for state in by_target.values()]
    return seen, sum(totals) if totals and all(t is not None for t in totals) else None


class Harvester:
    def __init__(self, platform: str):
        self.platform = platform
        self.comment_patterns = COMMENT_ENDPOINTS[platform]
        self.profile_patterns = PROFILE_ENDPOINTS.get(platform, [])
        self.patterns = self.comment_patterns + self.profile_patterns
        self.captured, self.ctx, self.pw, self.page = [], None, None, None
        self._coverage_cursor, self._seen_ids, self._reported_total = 0, set(), None
        self.target_url, self.max_comments, self.capture_session = None, None, None

    def start(self):
        self.pw  = sync_playwright().start()
        self.ctx = self.pw.chromium.launch_persistent_context(
            user_data_dir=str(PROFILE / self.platform), headless=False,
            viewport={"width": 1440, "height": 900},
            locale="id-ID", timezone_id="Asia/Jakarta",
        )
        self.page = self.ctx.pages[0] if self.ctx.pages else self.ctx.new_page()
        self.page.on("response", self._on_response)
        return self

    def _on_response(self, resp):
        if not any(p in resp.url for p in self.patterns): return
        if getattr(resp, "status", 200) >= 400: return
        ct = (resp.headers.get("content-type") or "")
        if "json" not in ct and "text" not in ct: return
        try: body = resp.text()
        except Exception: return
        # Most platforms return one JSON object per response — try that first, the common
        # case. Facebook (and some X GraphQL batch calls) can return several newline-delimited
        # JSON objects in one body, each optionally prefixed with the "for (;;);"
        # anti-JSON-hijacking header Facebook has used for over a decade; fall back to
        # per-line parsing only if the whole-body parse fails.
        payloads = []
        try:
            payloads.append(json.loads(body))
        except json.JSONDecodeError:
            for line in body.split("\n"):
                line = line.strip()
                if not line: continue
                if line.startswith("for (;;);"):
                    line = line[len("for (;;);"):]
                try: payloads.append(json.loads(line))
                except json.JSONDecodeError: continue
        kind = "comment" if any(p in resp.url for p in self.comment_patterns) else "profile"
        for payload in payloads:
            self.captured.append({"_platform": self.platform, "_url": resp.url,
                "_page_url": self.target_url or self.page.url, "_captured_at": now_utc(), "_kind": kind,
                "_capture_session": self.capture_session, "_max_comments": self.max_comments,
                "payload": payload})

    def open(self, url: str, max_comments=None):
        self.target_url, self.max_comments, self.capture_session = url, max_comments, now_utc()
        self._coverage_cursor, self._seen_ids, self._reported_total = len(self.captured), set(), None
        self.page.goto(url, wait_until="domcontentloaded", timeout=60_000)
        self.page.wait_for_timeout(3000)

    def _hover_comment_panel(self):
        """
        page.mouse.wheel() fires at the CURRENT mouse position, which defaults to (0, 0) —
        the page's top-left corner — until something moves it. On platforms with a dedicated
        side comment panel (TikTok, Instagram), a wheel event at (0, 0) mostly misses it
        entirely; hovering the panel first fixes that. Platforms with no such panel (X, and
        Facebook by default) just use the viewport-fraction fallback.
        """
        for sel in COMMENT_PANEL_SELECTORS.get(self.platform, []):
            try:
                panel = self.page.locator(sel).first
                panel.wait_for(state="attached", timeout=3000)
                box = panel.bounding_box()
                if box:
                    self.page.mouse.move(box["x"] + box["width"] / 2, box["y"] + box["height"] / 2)
                    return
            except Exception:
                continue
        vp = self.page.viewport_size or {"width": 1440, "height": 900}
        fx, fy = PANEL_FALLBACK.get(self.platform, (0.5, 0.5))
        self.page.mouse.move(vp["width"] * fx, vp["height"] * fy)

    def _wait_for_growth(self, timeout_ms: int, poll_ms: int = 150):
        """
        Waits up to timeout_ms for a new payload to arrive, but returns the moment one does
        instead of always sleeping the full window. Genuinely idle rounds (nothing new
        loading) still pay the full timeout_ms — that's necessary to distinguish "slow" from
        "actually plateaued".
        """
        start = len(self.captured)
        elapsed = 0
        while elapsed < timeout_ms:
            self.page.wait_for_timeout(poll_ms)
            elapsed += poll_ms
            if len(self.captured) > start:
                return True
        return False

    def scroll_comments(self, max_rounds: int = 200, idle_limit: int = 8, pause_ms: int = 1800):
        """
        Scrolls until capture growth stalls for `idle_limit` consecutive rounds, or
        `max_rounds` is hit — not a fixed count. Lazy-loading batch size varies per platform
        (TikTok ~20/XHR, others similar order of magnitude), so this is driven by growth, not
        a fixed comment-per-round assumption.
        """
        self._hover_comment_panel()
        stagnant, last = 0, self._coverage_snapshot()[0]
        for i in range(max_rounds):
            if self._at_cap(): break
            self.page.mouse.wheel(0, 3200)
            self._wait_for_growth(pause_ms)
            seen = self._coverage_snapshot()[0]
            if seen == last:
                stagnant += 1
                if stagnant >= idle_limit:
                    print(f"    plateaued after round {i} ({len(self.captured)} payloads captured)")
                    break
            else:
                stagnant, last = 0, self._coverage_snapshot()[0]
            if (i + 1) % 10 == 0:
                print(f"    round {i+1}/{max_rounds}: {len(self.captured)} payloads captured")
        return len(self.captured)

    def expand_replies(self, max_clicks: int = 300, pause_ms: int = 900):
        """
        Clicks every visible 'view replies'-style button so nested replies load through the
        same endpoints the harvester already listens for. Best-effort: button text/selectors
        vary by platform and locale, so REPLY_BUTTON_PATTERNS may need adjusting — especially
        for Facebook, the least stable of the four. Each click waits only until its reply
        payload actually arrives (see _wait_for_growth).
        """
        pattern = REPLY_BUTTON_PATTERNS.get(self.platform)
        if not pattern:
            return 0
        clicked = 0
        for _ in range(max_clicks):
            if self._at_cap(): break
            buttons = self.page.locator(f"text=/{pattern}/i")
            if buttons.count() == 0:
                break
            try:
                buttons.first.click(timeout=3000)
                clicked += 1
                self._wait_for_growth(pause_ms)
            except Exception:
                break
        if clicked:
            print(f"    expanded {clicked} reply thread(s)")
        return clicked

    def _coverage_snapshot(self):
        """Incremental unique comment candidates and an optional reported-total hint."""
        # Each newly captured payload is scanned once, not again on every scroll/batch.
        while self._coverage_cursor < len(self.captured):
            row = self.captured[self._coverage_cursor]
            self._coverage_cursor += 1
            ids, total = coverage_parts(row)
            self._seen_ids.update(ids)
            if total is not None: self._reported_total = max(self._reported_total or 0, total)
        return len(self._seen_ids), self._reported_total

    def _at_cap(self):
        return self.max_comments is not None and self._coverage_snapshot()[0] >= self.max_comments

    def scroll_until_coverage(self, target_pct: float = 0.95, batch_rounds: int = 60,
                               max_batches: int = 8, idle_limit: int = 12,
                               pause_ms: int = 1800, expand_replies: bool = True,
                               max_comments: int | None = 500):
        """Stop on a per-target cap, two stagnant batches, or the batch ceiling.

        target_pct is a compatibility argument only: reported totals are not a verified
        denominator for captured comments/replies. Canonical replay enforces the hard cap
        on normalized records, because an individual network response can overshoot it.
        """
        if max_comments is not None and max_comments < 0: raise ValueError("negative comment cap")
        self.max_comments = max_comments
        prev_seen, idle_batches = self._coverage_snapshot()[0], 0
        stop_reason = "max_batches exhausted"
        for batch in range(max_batches):
            self.scroll_comments(max_rounds=batch_rounds, idle_limit=idle_limit, pause_ms=pause_ms)
            if expand_replies:
                self.expand_replies(pause_ms=max(pause_ms // 2, 600))
                self.scroll_comments(max_rounds=30, idle_limit=6, pause_ms=pause_ms)
            seen, total = self._coverage_snapshot()
            pct_str = f"reported total hint: {total}" if total is not None else "total unknown"
            print(f"    batch {batch+1}/{max_batches}: {seen} comments ({pct_str})"
                  + (f" — cap {max_comments}" if max_comments else ""))
            if max_comments is not None and seen >= max_comments:
                stop_reason = f"reached max_comments cap ({max_comments})"
                break
            # Report totals as hints only: top-level, reply and filtered totals are not
            # guaranteed to have the same denominator as the captured unique-comment count.
            idle_batches = idle_batches + 1 if seen == prev_seen else 0
            if idle_batches >= 2:
                stop_reason = "plateaued — two batches in a row with no new unique comments"
                break
            prev_seen = seen
        seen, total = self._coverage_snapshot()
        self.stop_reason = stop_reason
        print(f"    stopped: {stop_reason}")
        return seen, total

    def flush(self) -> int:
        n = jsonl_append(RAW / f"{self.platform}_payloads.jsonl", self.captured)
        self.captured = []
        self._coverage_cursor = 0
        return n

    def close(self):
        if self.ctx: self.ctx.close()
        if self.pw:  self.pw.stop()


def run_browser_collection(targets: list[dict]):
    """
    Groups targets by platform so login happens at most once per platform per run.
    Every Harvester call is submitted to the same single-worker thread and .result()
    is awaited immediately, so from the notebook's point of view this still runs top
    to bottom in order — the threading is invisible except that it fixes the Windows
    subprocess issue. input() prompts happen on the main thread as normal and block
    exactly as long as it takes you to log in and get ready; that's expected, not a hang.
    Each platform in `targets` gets its own persistent browser_profile/<platform>/ and its
    own login pause the first time — TikTok, Instagram, X and Facebook logins are unrelated.
    """
    if not targets:
        print("no BROWSER_TARGETS configured — skipping"); return

    by_platform: dict[str, list[dict]] = {}
    for t in targets:
        by_platform.setdefault(t["platform"], []).append(t)

    with ThreadPoolExecutor(max_workers=1) as ex:
        for platform, items in by_platform.items():
            print(f"\n=== {platform} ({len(items)} target(s)) ===")
            h = ex.submit(lambda p=platform: Harvester(p).start()).result()
            session_ids = set()
            try:
                for i, t in enumerate(items):
                    ex.submit(h.open, t["url"], t.get("max_comments", 500)).result()
                    if i == 0:
                        input(f"  [{platform}] Log in if prompted, then press Enter here to continue... ")
                    print(f"  scrolling: {t['url']}")
                    seen, total = ex.submit(h.scroll_until_coverage,
                        t.get("target_coverage", 0.95), t.get("batch_rounds", 60),
                        t.get("max_batches", 8), t.get("idle_limit", 12),
                        t.get("pause_ms", 1800), t.get("expand_replies", True),
                        t.get("max_comments", 500)).result()
                    session_ids.update(h._seen_ids)
                    jsonl_append(RAW / "collection_runs.jsonl", [{
                        "platform": platform, "source_post_id": source_post_id(platform, t["url"]),
                        "capture_session": h.capture_session, "captured_unique": seen, "stop_reason": h.stop_reason,
                        "reported_total": total, "max_comments": t.get("max_comments", 500),
                        "collected_at": now_utc(), "sampling_method": "browser_display_order",
                    }])
                    ex.submit(h.flush).result()
                SESSION_SEEN[platform] = len(session_ids)
            finally:
                try: ex.submit(h.flush).result()
                finally: ex.submit(h.close).result()

run_browser_collection(BROWSER_TARGETS)

## 6. Parse the captured browser payloads

In [ ]:
def dig(obj, *paths, default=None):
    for path in paths:
        cur, ok = obj, True
        for key in path.split("."):
            if isinstance(cur, dict) and key in cur: cur = cur[key]
            elif isinstance(cur, list) and key.isdigit() and int(key) < len(cur): cur = cur[int(key)]
            else: ok = False; break
        if ok and cur is not None: return cur
    return default

def parse_tiktok(row):
    out = []
    stack = [(c, None) for c in reversed(dig(row["payload"], "comments", default=[]) or [])]
    seen = set()
    while stack:
        c, enclosing_parent = stack.pop()
        cid = as_id(c.get("cid"))
        if not cid or cid in seen: continue
        seen.add(cid)
        stack.extend((child, cid) for child in reversed(c.get("reply_comment") or []))
        u = c.get("user", {}) or {}
        reply_id = as_id(c.get("reply_id")) or enclosing_parent
        is_reply = bool(reply_id and reply_id != "0")
        # thread_id groups a top-level comment with all of its replies so coordination and
        # arrival-gap features mean the same thing on TikTok as on YouTube: a top-level
        # comment roots its own thread (thread_id = its own cid); a reply joins its parent's
        # thread (thread_id = reply_id = parent cid). The earlier version set top-level
        # comments to aweme_id (the whole video), which made every commenter on a video share
        # one giant "thread" — that overran coordination's max_group cap (silently zeroing
        # thread_co_* for TikTok) and made thread_id non-comparable across platforms.
        thread_id = reply_id if is_reply else c.get("cid")
        out.append(enrich_text_fields(empty_record(
            _platform="tiktok", _collected_at=row.get("_captured_at"), _source_url=row.get("_page_url"), _raw_ref=c.get("cid"),
            user_id=as_id(u.get("uid")) or as_id(u.get("sec_uid")), username=u.get("unique_id"),
            display_name=u.get("nickname"), bio_text=u.get("signature"),
            is_verified=(bool(u.get("custom_verify") or u.get("enterprise_verify_reason"))
                         if "custom_verify" in u or "enterprise_verify_reason" in u else None),
            has_custom_avatar=None,
            post_id=c.get("cid"), thread_id=thread_id,
            source_post_id=c.get("aweme_id") or source_post_id("tiktok", row.get("_page_url")),
            parent_comment_id=as_id(c.get("reply_to_reply_id")) or reply_id,
            created_at=timestamp_utc(c.get("create_time")),
            text_content=c.get("text"), like_count=c.get("digg_count"),
            reply_count=c.get("reply_comment_total"),
            is_reply=is_reply,
            is_repost=False, repost_count=None, source_device=None,
            media_types=["sticker"] if c.get("image_list") else ["text"])))
    return out

def parse_instagram(row):
    """
    Instagram's private-API comments endpoint (`/api/v1/media/<id>/comments/`) — the same shape
    tools like instaloader/instagrapi have relied on for years. Handles both the top-level
    "comments" list and the "child_comments" list returned when a reply thread is expanded.
    """
    out = []
    payload = row["payload"]
    endpoint_parent = re.search(r"/comments/([^/]+)/", row.get("_url", ""))
    endpoint_parent = endpoint_parent.group(1) if endpoint_parent else None
    items = [(c, None) for c in payload.get("comments", []) or []]
    items += [(c, endpoint_parent) for c in payload.get("child_comments", []) or []]
    seen = set()
    for c, enclosing_parent in items:
        cid = as_id(c.get("pk") or c.get("id"))
        if not cid or cid in seen: continue
        seen.add(cid)
        items.extend((child, cid) for child in c.get("preview_child_comments", []) or [])
        items.extend((child, cid) for child in c.get("child_comments", []) or [])
        u = c.get("user", {}) or {}
        parent_id = as_id(c.get("parent_comment_id")) or enclosing_parent
        is_reply = bool(parent_id)
        cid = c.get("pk") or c.get("id")
        thread_id = str(parent_id) if is_reply else str(cid or "")
        out.append(enrich_text_fields(empty_record(
            _platform="instagram", _collected_at=row.get("_captured_at"), _source_url=row.get("_page_url"), _raw_ref=cid,
            user_id=u.get("pk") or u.get("id"), username=u.get("username"),
            display_name=u.get("full_name"),
            is_verified=nullable_bool(u.get("is_verified")),
            # No verified default-avatar marker for Instagram's CDN paths — unlike TikTok
            # (where at least a stale marker exists), guessing one wrong would silently mark
            # real custom avatars as "default". Left null rather than guessed.
            has_custom_avatar=None,
            post_id=cid, thread_id=thread_id, parent_comment_id=parent_id,
            created_at=timestamp_utc(c.get("created_at")),
            text_content=c.get("text"), like_count=c.get("comment_like_count"),
            reply_count=c.get("child_comment_count"),
            is_reply=is_reply, is_repost=False, repost_count=None, source_device=None,
            media_types=["text"])))
    return out

def _parse_x_timestamp(s):
    if not s: return None
    try: return datetime.strptime(s, "%a %b %d %H:%M:%S %z %Y").isoformat()
    except (ValueError, TypeError): return None

def _iter_x_tweet_nodes(node):
    """
    Recursively finds every dict shaped like an X/Twitter GraphQL Tweet result, regardless of
    how deep the surrounding instructions/entries/itemContent wrapper nests it — that wrapper
    has changed shape before (TimelineAddEntries vs TimelineAddEntry, threaded conversations vs
    single-tweet detail) and searching structurally survives that better than hardcoding a path.
    """
    if isinstance(node, dict):
        legacy = node.get("legacy")
        if isinstance(legacy, dict) and "full_text" in legacy:
            yield node
            return
        elif isinstance(node.get("tweet"), dict) and isinstance(node["tweet"].get("legacy"), dict):
            yield node["tweet"]  # TweetWithVisibilityResults wrapper
            return
        for v in node.values():
            yield from _iter_x_tweet_nodes(v)
    elif isinstance(node, list):
        for v in node:
            yield from _iter_x_tweet_nodes(v)

def parse_x(row):
    """
    X's TweetDetail/ConversationTimeline GraphQL response already embeds the commenter's own
    profile stats (followers/following/bio/verified/account age) alongside every tweet, unlike
    TikTok/Instagram where the comment payload only has a thin user stub — so, unlike those two,
    X needs no separate profile-enrichment pass; the fields below are populated directly here.
    """
    out = []
    for t in _iter_x_tweet_nodes(row["payload"]):
        legacy = t.get("legacy", {}) or {}
        user_result = dig(t, "core.user_results.result", default={}) or {}
        u_legacy = user_result.get("legacy", {}) or {}
        tid = t.get("rest_id") or legacy.get("id_str")
        conv_id = legacy.get("conversation_id_str")
        target_id = source_post_id("x", row.get("_page_url"))
        if not target_id or str(tid) == target_id: continue
        if str(conv_id) != target_id and str(legacy.get("in_reply_to_status_id_str")) != target_id:
            continue
        is_reply = bool(legacy.get("in_reply_to_status_id_str"))
        # Keep the conversation root as thread_id and the direct reply target separately.
        # X conversation threads are coarser than top-level comment threads on other platforms.
        thread_id = conv_id or tid
        avatar = u_legacy.get("profile_image_url_https") or ""
        out.append(enrich_text_fields(empty_record(
            _platform="x", _collected_at=row.get("_captured_at"), _source_url=row.get("_page_url"), _raw_ref=tid,
            user_id=user_result.get("rest_id"), username=u_legacy.get("screen_name"),
            display_name=u_legacy.get("name"), bio_text=u_legacy.get("description"),
            account_created_at=_parse_x_timestamp(u_legacy.get("created_at")),
            is_verified=(bool(user_result.get("is_blue_verified") or u_legacy.get("verified"))
                         if "is_blue_verified" in user_result or "verified" in u_legacy else None),
            has_custom_avatar=("default_profile" not in avatar) if avatar else None,
            followers_count=u_legacy.get("followers_count"),
            following_count=u_legacy.get("friends_count"),
            total_posts_count=u_legacy.get("statuses_count"),
            post_id=tid, thread_id=thread_id, source_post_id=target_id,
            parent_comment_id=legacy.get("in_reply_to_status_id_str"),
            _profile_collected_at=row.get("_captured_at"), _profile_status="inline",
            created_at=_parse_x_timestamp(legacy.get("created_at")),
            text_content=legacy.get("full_text"),
            like_count=legacy.get("favorite_count"), reply_count=legacy.get("reply_count"),
            is_reply=is_reply, is_repost=bool(RE_RT.match(legacy.get("full_text") or "")),
            repost_count=legacy.get("retweet_count"), source_device=None,
            media_types=(["media"] if dig(legacy, "entities.media.0", default=None) else ["text"]))))
    return out

def _facebook_username_from_url(url):
    """Vanity slug from a facebook.com/<slug> profile URL, or None for a numeric profile.php
    URL — using the numeric id as a "username" would make handle_digit_suffix (a
    long-digit-run heuristic for auto-generated handles) fire on every such account, since
    Facebook's own URL scheme — not account age or provenance — is what put the digits there."""
    if not url: return None
    m = re.search(r"facebook\.com/([^/?]+)", url)
    if not m or m.group(1) in ("profile.php", "people"): return None
    return m.group(1)

def _iter_facebook_comment_nodes(node):
    """
    Facebook's GraphQL comment shape is the least stable of the four platforms here — doc_ids
    rotate constantly and Meta has restructured the UFI/comment payload multiple times. This
    searches structurally for any dict carrying a text body ('body'/'message', each usually
    {'text': ...}) alongside an 'author', rather than trusting one fixed path. Confirm this
    actually finds your comments with inspect_payloads('facebook') before trusting a run's
    output, and adjust the field names below to match what you see if it comes back empty.
    """
    if isinstance(node, dict):
        body = node.get("body") or node.get("message")
        text = body.get("text") if isinstance(body, dict) else None
        if (text is not None and isinstance(node.get("author"), dict)
                and (node.get("__typename") == "Comment" or "parent_comment" in node)) :
            yield node
        for v in node.values():
            yield from _iter_facebook_comment_nodes(v)
    elif isinstance(node, list):
        for v in node:
            yield from _iter_facebook_comment_nodes(v)

def parse_facebook(row):
    out = []
    for c in _iter_facebook_comment_nodes(row["payload"]):
        author = c.get("author", {}) or {}
        body = c.get("body") or c.get("message") or {}
        cid = c.get("id") or c.get("legacy_fbid")
        parent = dig(c, "parent_comment.id", default=None)
        is_reply = bool(parent)
        thread_id = str(parent) if is_reply else str(cid or "")
        created = c.get("created_time")
        out.append(enrich_text_fields(empty_record(
            _platform="facebook", _collected_at=row.get("_captured_at"), _source_url=row.get("_page_url"), _raw_ref=cid,
            user_id=author.get("id"), username=_facebook_username_from_url(author.get("url")),
            display_name=author.get("name"),
            is_verified=nullable_bool(author.get("is_verified")),
            post_id=cid, thread_id=thread_id, parent_comment_id=parent,
            created_at=timestamp_utc(created),
            text_content=body.get("text"),
            like_count=dig(c, "feedback.reaction_count.count", default=None),
            reply_count=dig(c, "feedback.replies_fields.total_count", default=None),
            is_reply=is_reply, is_repost=False, repost_count=None, source_device=None,
            media_types=["text"])))
    return out

PARSERS = {"tiktok": parse_tiktok, "instagram": parse_instagram, "x": parse_x, "facebook": parse_facebook}

def inspect_payloads(platform: str, n: int = 2):
    for row in jsonl_read(RAW / f"{platform}_payloads.jsonl")[:n]:
        print("URL:", row["_url"][:110])
        print(json.dumps(row["payload"], ensure_ascii=False)[:1200], "\n---")

def build_browser_canonical(platform: str):
    recs, errors, payload_count = {}, 0, 0
    session_ids = {}
    for row in jsonl_iter(RAW / f"{platform}_payloads.jsonl"):
        if row.get("_kind", "comment") != "comment": continue
        payload_count += 1
        try:
            for rec in PARSERS[platform](row):
                if rec.get("post_id"):
                    rec.update(_capture_session=row.get("_capture_session"),
                               _sampling_method="browser_display_order",
                               _requested_comment_cap=row.get("_max_comments"))
                    key = (row.get("_capture_session"), row.get("_page_url"))
                    seen = session_ids.setdefault(key, set())
                    cap = row.get("_max_comments")
                    if cap is not None and len(seen) >= cap and rec["post_id"] not in seen:
                        continue
                    seen.add(rec["post_id"])
                    recs[rec["post_id"]] = rec
        except (TypeError, ValueError, KeyError, AttributeError) as exc:
            errors += 1
            print(f"  ! parse error in payload {payload_count}: {type(exc).__name__}: {exc}")
    if errors:
        raise ValueError(f"{platform}: {errors} payloads failed; previous canonical preserved")
    if not recs:
        if payload_count: raise ValueError(f"{platform}: payloads captured but zero comments parsed")
        print(f"{platform}: no payloads captured"); return None
    df = to_frame(list(recs.values()))
    save_canonical(df, platform)
    print(f"{platform}: {len(df)} unique archived comments, {df.user_id.nunique()} accounts")
    return df

results_browser = {p: build_browser_canonical(p)
                   for p in sorted(set(t["platform"] for t in BROWSER_TARGETS))}

def coverage(platform: str):
    seen, total = estimate_coverage(jsonl_iter(RAW / f"{platform}_payloads.jsonl"))
    print(f"{platform}: {seen} archived unique comment candidates; reported total hint={total}; "
          "coverage percentage is unverified")

for p in sorted(set(t["platform"] for t in BROWSER_TARGETS)):
    coverage(p)

def _extract_profile_from_page(page):
    """
    Best-effort extraction of follower/following/post counts, verification, and bio from a
    TikTok profile page's own embedded state. TikTok server-renders this into a JSON script
    tag before any client-side API call fires, which is more reliable than waiting on a
    specific XHR that may or may not happen depending on how the page was reached. Tries
    known script-tag ids and JSON shapes; TikTok has changed both before and may again — if
    this stops finding data, inspect a profile page's script tags by hand (View Source ->
    search for '__UNIVERSAL_DATA_FOR_REHYDRATION__') and adjust the paths below.

    Note: "webapp.user-detail" below is ONE dict key containing a literal dot, not a nested
    path — TikTok's __DEFAULT_SCOPE__ object really is keyed that way, so this uses direct
    dict access rather than dig() (which would split "webapp.user-detail" into two levels).
    """
    for script_id in ("__UNIVERSAL_DATA_FOR_REHYDRATION__", "SIGI_STATE"):
        try:
            raw = page.locator(f"#{script_id}").text_content(timeout=2000)
        except Exception:
            continue
        if not raw:
            continue
        try:
            data = json.loads(raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(data, dict):
            continue

        default_scope = data.get("__DEFAULT_SCOPE__")
        if isinstance(default_scope, dict):
            user_info = default_scope.get("webapp.user-detail", {}).get("userInfo") \
                        if isinstance(default_scope.get("webapp.user-detail"), dict) else None
            if isinstance(user_info, dict) and (user_info.get("user") or user_info.get("stats")):
                return user_info.get("user") or {}, user_info.get("stats") or {}

        user_module = data.get("UserModule")
        if isinstance(user_module, dict):
            users_map = user_module.get("users")
            if isinstance(users_map, dict) and users_map:
                user = next(iter(users_map.values()), {}) or {}
                stats_map = user_module.get("stats") or {}
                stats = stats_map.get(user.get("uniqueId"), {}) if isinstance(stats_map, dict) else {}
                if user or stats:
                    return user, stats
    return {}, {}

PROFILE_FIELDS = ("followers_count", "following_count", "total_posts_count",
                   "is_verified", "bio_text", "_profile_collected_at", "_profile_status", "_profile_user_id", "_requested_user_id", *PROFILE_AUDIT_FIELDS)

def _load_profile_cache(platform: str) -> dict:
    """Reads raw/<platform>_profiles.jsonl into {username: {...}}, last entry per username wins
    (the file is append-only, so a re-enrichment of the same account appends a newer row
    rather than replacing the old one)."""
    cache = {}
    for row in jsonl_iter(RAW / f"{platform}_profiles.jsonl"):
        uname = row.get("username")
        if uname:
            cache[uname] = {k: row.get(k) for k in PROFILE_FIELDS}
            cache[uname]["profile_visibility"] = row.get("profile_visibility") or "unknown"
            cache[uname]["_profile_collected_at"] = row.get("_profile_collected_at") or row.get("_collected_at")
            cache[uname]["_profile_status"] = "cached" if profile_cache_fresh(cache[uname]) else "stale_cache"
    return cache

def profile_cache_fresh(profile, max_age_days=7):
    if profile.get("profile_access_status") not in {"readable", "restricted"}: return False
    timestamp = pd.to_datetime(profile.get("profile_checked_at"), utc=True, errors="coerce")
    if pd.isna(timestamp) or not profile.get("_profile_user_id"): return False
    age = (pd.Timestamp.now(tz="UTC")-timestamp).total_seconds()/86400
    return 0 <= age <= max_age_days

def _fetch_tiktok_profile(page, username, pause_ms=1500, expected_user_id=None):
    """One identity-checked visit; failed reads never establish private/public visibility."""
    expected_user_id = as_id(expected_user_id)
    try:
        response = page.goto(f"https://www.tiktok.com/@{username}", wait_until="domcontentloaded", timeout=30_000)
        if response is not None and response.status >= 400:
            raise ValueError(f"HTTP {response.status}")
        page.wait_for_timeout(pause_ms)
        user, stats = _extract_profile_from_page(page)
        user_id = as_id(user.get("id") or user.get("uid"))
        if (not user_id or str(user.get("uniqueId", "")).casefold() != username.casefold()
                or (expected_user_id is not None and user_id != expected_user_id)):
            raise ValueError("Profile identity unavailable or mismatched")
        checked = now_utc()
        return {
            "followers_count": stats.get("followerCount"), "following_count": stats.get("followingCount"),
            "total_posts_count": stats.get("videoCount"), "is_verified": nullable_bool(user.get("verified")),
            "bio_text": user.get("signature"), "_profile_collected_at": checked, "_profile_status": "observed",
            "_profile_user_id": user_id, "_requested_user_id": expected_user_id,
            **profile_audit(user, readable=True, checked_at=checked),
        }
    except Exception as exc:
        print(f"  ! {username}: {type(exc).__name__}: {exc}")
        return {"_requested_user_id": expected_user_id, "_profile_status": "failed", **profile_audit()}

def merge_profile_attempt(previous, attempt):
    """Retain old stats on failed checks only if their stable ID matches the requested ID."""
    if attempt.get("profile_access_status") != "failed": return attempt
    requested = as_id(attempt.get("_requested_user_id"))
    if requested and requested == as_id(previous.get("_profile_user_id")):
        return {**previous, **attempt, "_profile_status": "stale_cache"}
    return attempt

def profile_request_ids(frame):
    """Bind visits to observed stable accounts; ambiguous handles have no binding."""
    users = frame.dropna(subset=["username", "user_id"])[["username", "user_id"]].drop_duplicates()
    counts = users.groupby("username").size()
    return dict(users.loc[users.username.map(counts).eq(1)].itertuples(index=False, name=None))

def enrich_tiktok_profiles(usernames, pause_ms=1500, max_accounts=None, force_refresh=False, account_ids=None):
    todo = sorted({u for u in usernames if isinstance(u, str) and u})
    account_ids = account_ids or {}
    cached = _load_profile_cache("tiktok")
    to_visit = [u for u in todo if force_refresh or not profile_cache_fresh(cached.get(u, {}))
                or (as_id(account_ids.get(u)) is not None
                    and as_id(account_ids[u]) != as_id(cached.get(u, {}).get("_profile_user_id")))]
    if max_accounts is not None:
        if max_accounts < 0: raise ValueError("max_accounts must be nonnegative")
        to_visit = to_visit[:max_accounts]
    fresh = {}
    if to_visit:
        def _run():
            pw = sync_playwright().start()
            ctx = None
            results = {}
            try:
                ctx = pw.chromium.launch_persistent_context(
                    user_data_dir=str(PROFILE / "tiktok"), headless=False,
                    viewport={"width": 1440, "height": 900}, locale="id-ID", timezone_id="Asia/Jakarta")
                page = ctx.pages[0] if ctx.pages else ctx.new_page()
                for i, username in enumerate(to_visit):
                    attempt = _fetch_tiktok_profile(page, username, pause_ms, account_ids.get(username))
                    results[username] = merge_profile_attempt(cached.get(username, {}), attempt)
                    # Every attempted profile check, including failure, is durable immediately.
                    jsonl_append(RAW / "tiktok_profiles.jsonl", [{"username": username, **results[username]}])
                    if (i+1) % 25 == 0: print(f"  {i+1}/{len(to_visit)} profiles checked")
            finally:
                try:
                    if ctx: ctx.close()
                finally: pw.stop()
            return results
        with ThreadPoolExecutor(max_workers=1) as executor:
            fresh = executor.submit(_run).result()
    combined = {**cached, **fresh}
    result = {u: combined[u] for u in todo if u in combined}
    print(f"TikTok profiles: {len(fresh)} checked; {len(result)-len(fresh)} cached; "
          f"{sum(v.get('profile_access_status') == 'failed' for v in result.values())} failed outcomes")
    return result


def _extract_instagram_profile(payload, username=None):
    user = dig(payload, "data.user", "graphql.user", default={}) or {}
    if not user or (username and str(user.get("username", "")).casefold() != username.casefold()): return {}
    return {
        **profile_audit(user, readable=True, privacy_key="is_private"),
        "followers_count": dig(user, "edge_followed_by.count", default=None),
        "following_count": dig(user, "edge_follow.count", default=None),
        "total_posts_count": dig(user, "edge_owner_to_timeline_media.count", default=None),
        "is_verified": user.get("is_verified"),
        "bio_text": user.get("biography"),
        "_profile_collected_at": now_utc(), "_profile_status": "observed",
                            "_profile_user_id": as_id(user.get("id") or user.get("pk")),
    }

def enrich_instagram_profiles(usernames, pause_ms: int = 1500, max_accounts: int | None = None,
                               force_refresh: bool = False):
    """
    Same cache-aware, same-persistent-profile pattern as enrich_tiktok_profiles above, but
    reads a clean network JSON response instead of scraping an embedded DOM script tag:
    visiting instagram.com/<username>/ triggers a request matching
    PROFILE_ENDPOINTS['instagram'], which the Harvester's own response listener (§5) already
    captures — this just drives that same Harvester across the profile pages. Best-effort like
    the TikTok version: Instagram has changed this response shape before and may again — if
    n_have stays 0, inspect_payloads won't help here (it only reads *_payloads.jsonl, not
    *_profiles.jsonl) — instead print a captured row's payload by hand to see the current shape.

    Requires a valid session in browser_profile/instagram — i.e. this run (or an earlier one)
    already completed the §5 login pause for Instagram.
    """
    todo = sorted({u for u in usernames if u})
    if not todo:
        print("no usernames to enrich"); return {}

    cached = _load_profile_cache("instagram")
    to_visit = [u for u in todo if force_refresh or not profile_cache_fresh(cached.get(u, {}))]
    if max_accounts is not None:
        if max_accounts < 0: raise ValueError("max_accounts must be nonnegative")
        to_visit = to_visit[:max_accounts]

    fresh = {}
    if to_visit:
        def _run():
            h = Harvester("instagram").start()
            results = {}
            print(f"enriching {len(to_visit)} Instagram profile(s) "
                  f"({len(todo) - len(to_visit)} already cached, reused)...")
            for i, uname in enumerate(to_visit):
                try:
                    h.captured = []
                    h.open(f"https://www.instagram.com/{uname}/")
                    h._wait_for_growth(pause_ms)
                    for row in h.captured:
                        if row.get("_kind") == "profile":
                            fields = _extract_instagram_profile(row["payload"], uname)
                            if fields:
                                results[uname] = fields
                                break
                except Exception as e:
                    print(f"  ! {uname}: {type(e).__name__}: {e}")
                if (i + 1) % 25 == 0:
                    print(f"  {i+1}/{len(to_visit)} profiles visited, {len(results)} succeeded so far")
            h.close()
            return results

        with ThreadPoolExecutor(max_workers=1) as ex:
            fresh = ex.submit(_run).result()
        if fresh:
            jsonl_append(RAW / "instagram_profiles.jsonl",
                         [{"username": k, **v, "_collected_at": now_utc()} for k, v in fresh.items()])
    else:
        print("No Instagram profile visits scheduled (cache/budget); missing or stale profiles remain explicit")

    combined = {**cached, **fresh}
    result = {u: combined[u] for u in todo if u in combined}
    print(f"profiles: {len(fresh)} newly visited + {len(result) - len(fresh)} reused from cache "
          f"= {len(result)}/{len(todo)} total")
    return result

def merge_profile_enrichment(df: pd.DataFrame, profiles: dict) -> pd.DataFrame:
    """Profile-page values are more authoritative than anything guessed from the comment
    payload (e.g. is_verified there only catches enterprise verification), so they overwrite
    the existing column where available rather than just filling nulls."""
    if not profiles or "username" not in df.columns:
        return df
    df = df.copy()
    profile_ids = df["username"].map(lambda u: as_id(profiles.get(u, {}).get("_profile_user_id")))
    identity_matches = profile_ids.notna() & profile_ids.eq(df["user_id"].map(as_id))
    for col in PROFILE_FIELDS:
        enriched = df["username"].map(lambda u: profiles.get(u, {}).get(col))
        if col == "_requested_user_id": continue  # cache-only request provenance
        if col in PROFILE_AUDIT_FIELDS:
            requested = df["username"].map(lambda u: as_id(profiles.get(u, {}).get("_requested_user_id")))
            failed = df["username"].map(lambda u: profiles.get(u, {}).get("profile_access_status") == "failed")
            bound_failure = failed & requested.notna() & requested.eq(df["user_id"].map(as_id))
            enriched = enriched.where(identity_matches | bound_failure)
        else:
            enriched = enriched.where(identity_matches)
        df[col] = enriched.where(enriched.notna(), df[col]) if col in df else enriched
    return df

# Profile enrichment is not needed for the three-column scraping export.


## 7. Combine three-column scraping exports

The legacy feature functions below remain available for older full-schema datasets; this cell now runs `assemble_scraped_data()` only.


In [ ]:
ACCOUNT_KEY = "global_user_id"

def _coerce_bool(series):
    return series.map(nullable_bool).astype("boolean")

def prepare_comments(df):
    """Validate identity, remove repeat observations, and recover source IDs for legacy CSVs."""
    d = df.copy()
    for field in CANONICAL_FIELDS:
        if field not in d:
            d[field] = [[] for _ in range(len(d))] if field in LIST_FIELDS else None
    d["profile_visibility"] = d["profile_visibility"].fillna("unknown")
    for field in IDENTITY_FIELDS:
        d[field] = d[field].map(as_id).astype("string")
    d["source_post_id"] = d["source_post_id"].fillna(pd.Series(
        [source_post_id(p, u) for p, u in zip(d["_platform"], d["_source_url"])], index=d.index))
    # Authorless/deleted comments are kept in canonical files, but are not one synthetic account.
    valid = d["user_id"].notna() & d["post_id"].notna() & d["_platform"].notna()
    if not valid.all(): print(f"Excluded {(~valid).sum()} comments without account/comment identity")
    d = d.loc[valid].copy()
    d["_observed"] = pd.to_datetime(d["_collected_at"], utc=True, errors="coerce", format="mixed")
    d = d.sort_values("_observed", na_position="first", kind="stable").drop_duplicates(
        ["_platform", "post_id"], keep="last").drop(columns="_observed").reset_index(drop=True)
    d[ACCOUNT_KEY] = d["_platform"].astype("string") + ":" + d["user_id"]
    return d

def derive_features(df, window_days=None):
    d = prepare_comments(df)
    observed_text = d["_text_observed"].map(nullable_bool)
    fallback_text = d["text_content"].map(lambda value: isinstance(value, str) and bool(value.strip()))
    d["_text_available"] = observed_text.where(observed_text.notna(), fallback_text).astype(bool)
    d["created_at_dt"] = pd.to_datetime(d["created_at"], utc=True, errors="coerce", format="mixed")
    if window_days is not None:
        if not np.isfinite(window_days) or window_days <= 0: raise ValueError("window_days must be positive")
        span = window_days
    else:
        # A sampling exposure proxy, computed separately by platform, not a full activity history.
        g = d.groupby("_platform")["created_at_dt"]
        span = (g.transform("max")-g.transform("min")).dt.total_seconds().div(86400).clip(lower=1)
    d["observed_activity_rate"] = d.groupby(ACCOUNT_KEY)["post_id"].transform("size") / span
    d["_dupe_key"] = d["text_content"].map(dedupe_key)
    valid = d["_dupe_key"].ne("")
    for col, keys, op, value in [
        ("duplicate_text_count", [ACCOUNT_KEY, "_dupe_key"], "size", "post_id"),
        ("corpus_duplicate_count", ["_platform", "_dupe_key"], "size", "post_id"),
        ("distinct_users_same_text", ["_platform", "_dupe_key"], "nunique", ACCOUNT_KEY),
    ]:
        d[col] = d.loc[valid].groupby(keys)[value].transform(op).reindex(d.index).fillna(0).astype(int)
    d["is_repeated_text"] = d["duplicate_text_count"].gt(1) & valid
    # Short generic reactions are not sufficient evidence of coordinated messaging.
    substantial = d["text_content"].fillna("").str.count(r"\w").ge(12)
    d["is_shared_text"] = d["distinct_users_same_text"].gt(1) & substantial
    d["text_length"] = d["clean_text"].fillna("").str.len()
    for col, field in [("hashtag_count", "hashtags"), ("mention_count", "user_mentions"), ("url_count", "urls")]:
        d[col] = d[field].map(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)
    d["media_count"] = d["media_types"].map(
        lambda x: sum(v != "text" for v in x) if isinstance(x, (list, tuple)) else 0)
    for col in ["followers_count", "following_count", "total_posts_count", "like_count", "reply_count", "repost_count"]:
        d[col] = pd.to_numeric(d[col], errors="coerce")
        d[col] = d[col].where(d[col].ge(0) & np.isfinite(d[col]))
    d["follower_to_following_ratio"] = (d["followers_count"]+1)/(d["following_count"]+1)
    d["log_follower_following_ratio"] = np.log1p(d["followers_count"])-np.log1p(d["following_count"])
    d["bio_length"] = d["bio_text"].astype("string").str.len().astype("Float64")
    observed = pd.to_datetime(d["_collected_at"], utc=True, errors="coerce", format="mixed")
    created = pd.to_datetime(d["account_created_at"], utc=True, errors="coerce", format="mixed")
    d["account_age_days"] = (observed-created).dt.total_seconds()/86400
    d["account_age_days"] = d["account_age_days"].where(d["account_age_days"].ge(0))
    for col in ("is_reply", "is_verified", "is_repost"):
        d[col] = _coerce_bool(d[col])
    d["handle_digit_suffix"] = d["username"].astype("string").str.contains(r"\d{6,}$").astype("Int64")
    return d.drop(columns="_dupe_key")

def _group_coordination(df, group_col, prefix, min_shared, max_group, max_pair_events):
    """Exact co-occurrence where affordable; skipped work is explicitly marked missing."""
    keys = ["_platform", "source_post_id"]
    if group_col != "source_post_id": keys.append(group_col)
    ut = df[[ACCOUNT_KEY]+keys].dropna().drop_duplicates()
    groups = ut.groupby(keys, sort=True)[ACCOUNT_KEY]
    memberships = ut.groupby(ACCOUNT_KEY).size()
    out = pd.DataFrame({ACCOUNT_KEY: df[ACCOUNT_KEY].drop_duplicates()})
    out[f"{prefix}_groups"] = out[ACCOUNT_KEY].map(memberships).fillna(0).astype(int)
    pairs, skipped, events = Counter(), Counter(), 0
    # Accounts seen in < min_shared groups cannot possibly form a qualifying pair.
    eligible = set(memberships[memberships >= min_shared].index)
    for _, users in groups:
        all_users = list(users)
        candidates = sorted(u for u in all_users if u in eligible)
        cost = len(candidates)*(len(candidates)-1)//2
        if len(candidates) > max_group or events+cost > max_pair_events:
            skipped.update(all_users)
            continue
        events += cost
        pairs.update(combinations(candidates, 2))
    partners, strength = Counter(), Counter()
    for (a,b), count in pairs.items():
        if count >= min_shared:
            partners.update([a,b])
            strength[a] = max(strength[a], count); strength[b] = max(strength[b], count)
    out[f"{prefix}_skipped_groups"] = out[ACCOUNT_KEY].map(skipped).fillna(0).astype(int)
    incomplete = out[f"{prefix}_skipped_groups"].gt(0) | out[f"{prefix}_groups"].eq(0)
    out[f"{prefix}_partners"] = out[ACCOUNT_KEY].map(partners).fillna(0).mask(incomplete)
    out[f"{prefix}_max"] = out[ACCOUNT_KEY].map(strength).fillna(0).mask(incomplete)
    out[f"{prefix}_rate"] = out[f"{prefix}_max"]/out[f"{prefix}_groups"].replace(0, np.nan)
    return out

def coordination_features(df, min_shared=2, max_group=1000, max_pair_events=2_000_000):
    if min_shared < 1 or max_group < 1 or max_pair_events < 0: raise ValueError("invalid coordination budget")
    out = _group_coordination(df, "thread_id", "thread_co", min_shared, max_group, max_pair_events)
    out = out.merge(_group_coordination(df, "source_post_id", "post_co", min_shared, max_group,
                                      max_pair_events), on=ACCOUNT_KEY, validate="one_to_one")
    keys = ["_platform", "source_post_id", "thread_id"]
    d = df.dropna(subset=keys).copy()
    d["ts"] = pd.to_datetime(d["created_at"], errors="coerce", utc=True, format="mixed")
    d = d.dropna(subset=["ts"]).sort_values(keys+["ts", ACCOUNT_KEY], kind="stable")
    g = d.groupby(keys)
    # This describes neighboring observed comments, not pairwise coordination proof.
    d["gap"] = g["ts"].diff().dt.total_seconds().where(g[ACCOUNT_KEY].shift().ne(d[ACCOUNT_KEY]))
    sync = d.groupby(ACCOUNT_KEY)["gap"].median().rename("median_thread_arrival_gap")
    return out.merge(sync, on=ACCOUNT_KEY, how="left", validate="one_to_one")

ID_COLUMNS = [*PROFILE_AUDIT_FIELDS, "global_user_id", "user_id", "username", "display_name", "_platform", "post_id",
              "source_post_id", "parent_comment_id", "thread_id", "created_at", "text_content", "bio_text",
              "_collected_at", "_profile_collected_at", "_profile_status", "_source_url", "_raw_ref",
              "_collector_version", "_capture_session", "_sampling_method", "_requested_comment_cap"]
FEATURE_COLUMNS = [
    "hashtag_count", "mention_count", "url_count", "media_count", "is_reply", "text_length", "language",
    "duplicate_text_count", "corpus_duplicate_count", "distinct_users_same_text", "is_repeated_text", "is_shared_text",
    "thread_co_partners", "thread_co_rate", "thread_co_skipped_groups", "median_thread_arrival_gap",
    "post_co_partners", "post_co_rate", "post_co_skipped_groups", "observed_activity_rate",
    "followers_count", "following_count", "total_posts_count", "follower_to_following_ratio",
    "log_follower_following_ratio", "is_verified", "bio_length", "handle_digit_suffix", "account_age_days",
]
# Schema-backed account columns: old names remain audit/compatibility columns only.
BASELINE_SCHEMA = json.loads(Path("behavioral_baseline.schema.json").read_text(encoding="utf-8"))
BASELINE_PROPERTIES = BASELINE_SCHEMA["$defs"]["features"]["properties"]
BASELINE_RENAME = {name: spec["x-notebook-column"] for name, spec in BASELINE_PROPERTIES.items()}
BASELINE_ACCOUNT_FEATURE_COLUMNS = list(BASELINE_PROPERTIES)
LEGACY_BASELINE_ACCOUNT_FEATURE_COLUMNS = list(BASELINE_RENAME.values())
BASELINE_STATUS_VALUES = {"observed", "structural_missing", "collection_failed", "insufficient_support", "computation_skipped", "unknown"}

def standardize_baseline(accounts, comments, platform_missingness=None):
    """Apply availability/support gates; keep observed values separate from model imputation."""
    out = accounts.copy()
    for name, legacy in BASELINE_RENAME.items():
        out[name] = pd.to_numeric(out[legacy], errors="coerce")
        out[name + "__status"] = np.where(out[name].notna(), "observed", "unknown")
    g = comments.groupby(ACCOUNT_KEY)
    support = pd.DataFrame({
        "text_observation_count": comments["_text_available"].groupby(comments[ACCOUNT_KEY]).sum(),
        "reply_status_count": g["is_reply"].count(),
        "source_id_count": g["source_post_id"].count(),
    })
    for col in support:
        out[col] = out[ACCOUNT_KEY].map(support[col]).fillna(0).astype(int)

    def missing(mask, names, reason):
        for name in names:
            out.loc[mask, name] = np.nan
            out.loc[mask, name + "__status"] = reason

    text_features = ["mean_comment_length_chars", "mean_hashtags_per_comment", "mean_mentions_per_comment",
                     "mean_urls_per_comment", "repeated_comment_ratio", "shared_comment_ratio"]
    missing(out.text_observation_count.lt(out.n_comments), text_features, "unknown")
    # Incomplete corpus inputs affect other accounts' shared-text/peer denominators as well.
    bad_text_platforms = set(comments.loc[~comments["_text_available"], "_platform"])
    missing(out._platform.isin(bad_text_platforms), ["shared_comment_ratio"], "unknown")
    missing(out.reply_status_count.eq(0), ["reply_ratio"], "unknown")
    missing(out.n_source_posts.eq(0), ["source_post_count"], "unknown")
    peers = ["recurring_peer_count", "max_peer_overlap_ratio"]
    missing(out.n_source_posts.lt(2), peers, "insufficient_support")
    bad_source_platforms = set(comments.loc[comments.source_post_id.isna(), "_platform"])
    missing(out._platform.isin(bad_source_platforms), peers, "unknown")
    missing(out.post_co_skipped_groups.gt(0), peers, "computation_skipped")
    missing(out.n_valid_timestamps.lt(2), ["median_comment_interval_seconds"], "insufficient_support")
    missing(out.n_timing_intervals.lt(3) | out.intercomment_burstiness.isna(),
            ["comment_interval_burstiness"], "insufficient_support")

    for platform, overrides in (platform_missingness or {}).items():
        if platform not in {"tiktok", "instagram", "x", "facebook", "youtube"}:
            raise ValueError(f"Unknown platform: {platform}")
        for name, reason in overrides.items():
            if name not in BASELINE_RENAME or reason not in BASELINE_STATUS_VALUES - {"observed"}:
                raise ValueError(f"Invalid missingness override: {platform}/{name}/{reason}")
            missing(out._platform.eq(platform), [name], reason)
    for name, spec in BASELINE_PROPERTIES.items():
        out[name] = out[name].astype("Int64" if "integer" in spec["type"] else "Float64")
    validate_baseline(out)
    return out

def validate_baseline(frame):
    for name, spec in BASELINE_PROPERTIES.items():
        values = pd.to_numeric(frame[name], errors="raise")
        valid = values.notna()
        if not np.isfinite(values[valid].astype(float)).all(): raise ValueError(f"Nonfinite {name}")
        if values[valid].lt(spec.get("minimum", -np.inf)).any() or values[valid].gt(spec.get("maximum", np.inf)).any():
            raise ValueError(f"Out-of-range {name}")
        if "integer" in spec["type"] and values[valid].mod(1).ne(0).any():
            raise ValueError(f"Noninteger {name}")
        status = frame[name + "__status"]
        if not status.isin(BASELINE_STATUS_VALUES).all() or not status.eq("observed").eq(valid).all():
            raise ValueError(f"Value/status mismatch for {name}")

def baseline_quality_report(accounts):
    rows = []
    for platform, group in accounts.groupby("_platform"):
        for name in BASELINE_ACCOUNT_FEATURE_COLUMNS:
            row = {"platform": platform, "feature": name, "account_count": len(group),
                   "observed_count": int(group[name].notna().sum()),
                   "null_fraction": group[name].isna().mean()}
            row.update({reason + "_count": int(group[name + "__status"].eq(reason).sum())
                        for reason in sorted(BASELINE_STATUS_VALUES)})
            rows.append(row)
    return pd.DataFrame(rows)

def baseline_export(accounts, window_start=None, window_end=None):
    columns = [*PROFILE_AUDIT_FIELDS, ACCOUNT_KEY, "_platform", "window_start", "window_end"] + BASELINE_ACCOUNT_FEATURE_COLUMNS
    columns += [name + "__status" for name in BASELINE_ACCOUNT_FEATURE_COLUMNS]
    columns += ["text_observation_count", "reply_status_count", "source_id_count", "n_valid_timestamps", "n_timing_intervals"]
    out = accounts.copy()
    out["window_start"], out["window_end"] = window_start, window_end
    return out[columns]

def baseline_json_records(frame):
    """Strict schema records require explicitly configured observation-window boundaries."""
    validate_baseline(frame)
    for row in frame.to_dict("records"):
        start, end = row.get("window_start"), row.get("window_end")
        if start is None or end is None or pd.isna(start) or pd.isna(end):
            raise ValueError("Set OBSERVATION_WINDOW_START and OBSERVATION_WINDOW_END for schema JSONL")
        start, end = pd.to_datetime(start, utc=True), pd.to_datetime(end, utc=True)
        if start >= end: raise ValueError("Observation window must have start < end")
        features = {}
        for name, spec in BASELINE_PROPERTIES.items():
            value = row[name]
            features[name] = None if pd.isna(value) else int(value) if "integer" in spec["type"] else float(value)
        yield {"schema_version": "1.0.0", "account_id": row[ACCOUNT_KEY], "platform": row["_platform"],
               "window_start": start.isoformat(), "window_end": end.isoformat(), "features": features,
               "feature_status": {name: row[name + "__status"] for name in features},
               **{name: None if pd.isna(row.get(name)) else row[name] for name in PROFILE_AUDIT_FIELDS}}

PROFILE_ACCOUNT_FEATURE_COLUMNS = [
    "followers_count", "following_count", "total_posts_count", "log_follower_following_ratio",
    "is_verified", "bio_length", "handle_digit_suffix", "account_age_days",
]
ACCOUNT_FEATURE_COLUMNS = list(dict.fromkeys(BASELINE_ACCOUNT_FEATURE_COLUMNS + PROFILE_ACCOUNT_FEATURE_COLUMNS + [
    "text_length_max", "hashtag_count_max", "mention_count_max", "url_count_max", "media_count_mean",
    "media_count_max", "duplicate_text_count_max", "corpus_duplicate_count_max", "distinct_users_same_text_max",
    "thread_co_partners", "thread_co_rate", "median_thread_arrival_gap", "observed_activity_rate",
    "follower_to_following_ratio", "thread_co_skipped_groups", "post_co_skipped_groups", "n_valid_timestamps",
    "n_timing_intervals", "observed_span_days",
]))

def aggregate_accounts(df, platform_missingness=None):
    d = df.copy()
    d["is_reply"] = _coerce_bool(d["is_reply"])
    g = d.groupby(ACCOUNT_KEY, sort=False)
    acc = pd.DataFrame(index=g.size().index)
    for col in ["text_length", "hashtag_count", "mention_count", "url_count", "media_count"]:
        acc[f"{col}_mean"] = g[col].mean(); acc[f"{col}_max"] = g[col].max()
    acc["reply_fraction"] = g["is_reply"].mean()
    acc["repeated_text_fraction"] = g["is_repeated_text"].mean()
    acc["shared_text_fraction"] = g["is_shared_text"].mean()
    for col in ["duplicate_text_count", "corpus_duplicate_count", "distinct_users_same_text"]:
        acc[f"{col}_max"] = g[col].max()
    for col in ["thread_co_partners", "thread_co_rate", "median_thread_arrival_gap", "observed_activity_rate",
                "post_co_partners", "post_co_rate", "thread_co_skipped_groups", "post_co_skipped_groups"]:
        acc[col] = g[col].first()
    # Select one actual latest profile snapshot; groupby.first() synthesizes a mix of dates.
    profile_time = pd.to_datetime(d["_profile_collected_at"], utc=True, errors="coerce", format="mixed")
    capture_time = pd.to_datetime(d["_collected_at"], utc=True, errors="coerce", format="mixed")
    latest = d.assign(_snapshot=profile_time.fillna(capture_time)).sort_values(
        "_snapshot", na_position="first", kind="stable").drop_duplicates(ACCOUNT_KEY, keep="last").set_index(ACCOUNT_KEY)
    for col in PROFILE_ACCOUNT_FEATURE_COLUMNS + ["follower_to_following_ratio", "_platform", "user_id", "username",
                                                "display_name", "_profile_collected_at", "_profile_status"]:
        acc[col] = latest[col]
    # A failed check may be newer than the last successfully collected profile statistics.
    audit_time = pd.to_datetime(d["profile_checked_at"], utc=True, errors="coerce", format="mixed")
    latest_audit = d.assign(_audit_time=audit_time).sort_values(
        "_audit_time", na_position="first", kind="stable").drop_duplicates(ACCOUNT_KEY, keep="last").set_index(ACCOUNT_KEY)
    for col in PROFILE_AUDIT_FIELDS: acc[col] = latest_audit[col]
    acc["n_comments"] = g.size()
    acc["n_source_posts"] = g["source_post_id"].nunique()
    acc["n_videos"] = acc["n_source_posts"]  # compatibility alias, not a second model feature
    acc["dominant_language"] = g["language"].agg(lambda s: Counter(s.dropna()).most_common(1)[0][0] if s.notna().any() else None)
    timing = d.dropna(subset=["created_at_dt"]).sort_values([ACCOUNT_KEY, "created_at_dt"], kind="stable")
    timing["gap"] = timing.groupby(ACCOUNT_KEY)["created_at_dt"].diff().dt.total_seconds()
    tg = timing.groupby(ACCOUNT_KEY)
    acc["n_valid_timestamps"] = tg.size().reindex(acc.index, fill_value=0)
    gaps = tg["gap"].agg(["count", "median", "mean", "std"])
    acc["n_timing_intervals"] = gaps["count"].reindex(acc.index, fill_value=0)
    acc["median_intercomment_seconds"] = gaps["median"]
    acc["intercomment_burstiness"] = ((gaps["std"]-gaps["mean"])/(gaps["std"]+gaps["mean"]).replace(0,np.nan)).where(gaps["count"] >= 3)
    acc["observed_span_days"] = (tg["created_at_dt"].max()-tg["created_at_dt"].min()).dt.total_seconds()/86400
    return standardize_baseline(acc.reset_index(), d, platform_missingness)

def feature_quality_report(accounts):
    rows = []
    for platform, group in accounts.groupby("_platform"):
        for col in ACCOUNT_FEATURE_COLUMNS:
            rows.append({"platform": platform, "column": col, "accounts": len(group),
                         "missing_fraction": group[col].isna().mean(),
                         "n_unique_nonmissing": group[col].nunique(dropna=True),
                         "constant_or_empty": group[col].nunique(dropna=True) <= 1})
    return pd.DataFrame(rows)

def assemble(platforms=("youtube", "tiktok", "instagram", "x", "facebook"), window_start=None, window_end=None, platform_missingness=None):
    frames = [load_canonical(p) for p in platforms if (CANONICAL / f"{p}.csv").exists()]
    if not frames: raise FileNotFoundError("no canonical CSV files — nothing was collected")
    raw = pd.concat(frames, ignore_index=True)
    if (window_start is None) != (window_end is None): raise ValueError("Configure both observation-window boundaries")
    if window_start is not None:
        start, end = pd.to_datetime(window_start, utc=True), pd.to_datetime(window_end, utc=True)
        if pd.isna(start) or pd.isna(end) or start >= end: raise ValueError("Invalid observation window")
        timestamps = pd.to_datetime(raw["created_at"], utc=True, errors="coerce", format="mixed")
        raw = raw.loc[timestamps.ge(start) & timestamps.lt(end)].copy()
        window_start, window_end = start.isoformat(), end.isoformat()
    df = derive_features(raw)
    if df.empty: raise ValueError("no comments with valid account and comment IDs")
    df = df.merge(coordination_features(df), on=ACCOUNT_KEY, how="left", validate="many_to_one")
    write_csv_with_lists(df[ID_COLUMNS+FEATURE_COLUMNS], FEATURES / "dataset.csv")
    accounts = aggregate_accounts(df, platform_missingness)
    write_csv_with_lists(accounts, FEATURES / "dataset_accounts.csv")
    baseline = baseline_export(accounts, window_start, window_end)
    write_csv_with_lists(baseline, FEATURES / "baseline_accounts.csv")
    baseline_quality_report(accounts).to_csv(FEATURES / "baseline_feature_quality.csv", index=False)
    json_path = FEATURES / "baseline_accounts.jsonl"
    # Always replace the strict export so a previous window is never mistaken for this run.
    with json_path.open("w", encoding="utf-8") as handle:
        if window_start is not None:
            for record in baseline_json_records(baseline):
                handle.write(json.dumps(record, ensure_ascii=False, allow_nan=False) + "\n")
        else:
            print("Baseline CSV exported for exploration; set both window boundaries for schema JSONL")
    feature_quality_report(accounts).to_csv(FEATURES / "feature_quality.csv", index=False)
    print(f"comments: {len(df)}; accounts: {len(accounts)}; source posts: {df.groupby('_platform').source_post_id.nunique().to_dict()}")
    return df

def assemble_scraped_data(platforms=("youtube", "tiktok", "instagram", "x", "facebook")):
    """Combine available platform CSVs in the same three-column format."""
    frames = [scraping_columns(load_canonical(p)) for p in platforms
              if (CANONICAL / f"{p}.csv").exists()]
    if not frames:
        raise FileNotFoundError("no canonical CSV files - nothing was collected")
    out = pd.concat(frames, ignore_index=True)
    write_csv_with_lists(out, FEATURES / "dataset.csv")
    return out


df = assemble_scraped_data()
print(f"Exported {len(df)} comments with columns: {list(df.columns)}")
df.head()


## 7b. Train-only baseline preprocessing and model factories

The account export now includes the 13 standardized names from `behavioral_baseline.schema.json`.
`baseline_accounts.csv` contains these features plus identity/window/support and `__status` metadata.
Legacy column names remain in the wider account export for compatibility. Neither the status columns
nor identifiers enter the behavioral matrix. Set both UTC observation-window boundaries in §2 for
strict schema JSONL; otherwise the CSV is explicitly an exploratory archive aggregate.

`shared_median` drops features lacking support on any training platform, then log-transforms,
median-imputes and standardizes. `platform_median` fits per-platform medians; missing platform medians
raise unless `allow_pooled_fallback=True` explicitly enables the pooled training median.
`native` preserves NaNs for histogram gradient boosting. All three drop all-null/constant training
features. The linear modes add fixed binary missingness indicators outside the 13-feature schema.

Fit only on labeled training rows, after rebuilding sharing/coordination features within each
fold's permitted corpus and time cutoff. The window filter uses comment event time; it is not an
as-of snapshot filter. For historical prediction, also exclude captures obtained after that cutoff.
No model is automatically trained on the full export. See `BASELINE_SCHEMA.md` for the full policy.


In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

class BaselinePreprocessor(TransformerMixin, BaseEstimator):
    """Fit selection/imputation/scaling on one training fold, never on the exported corpus.

    shared_median: common supported columns + pooled training medians.
    platform_median: training-platform medians; pooled fallback requires explicit opt-in.
    native: keep NaNs for a missing-value-aware model. No scaling or imputation.
    """
    def __init__(self, strategy="shared_median", min_observations=1,
                 allow_pooled_fallback=False, add_missing_indicators=True):
        self.strategy = strategy
        self.min_observations = min_observations
        self.allow_pooled_fallback = allow_pooled_fallback
        self.add_missing_indicators = add_missing_indicators

    def _read(self, X):
        if not isinstance(X, pd.DataFrame) or "_platform" not in X or X._platform.isna().any():
            raise ValueError("Pass a baseline DataFrame with complete _platform metadata")
        validate_baseline(X)
        return X[BASELINE_ACCOUNT_FEATURE_COLUMNS].astype(float)

    def _values(self, X):
        values = self._read(X)[self.selected_features_].copy()
        if self.strategy != "native":
            values[self.log_features_] = np.log1p(values[self.log_features_])
        return values

    def _fill(self, values, platforms):
        out = values.copy()
        if self.strategy == "platform_median":
            for name in self.selected_features_:
                fills = platforms.map(self.platform_medians_[name])
                out[name] = out[name].fillna(fills)
            if out.isna().any().any() and not self.allow_pooled_fallback:
                raise ValueError("No supported training-platform median; use shared_median/native or explicitly allow pooled fallback")
        return out.fillna(self.medians_)

    def fit(self, X, y=None):
        if self.strategy not in {"shared_median", "platform_median", "native"}:
            raise ValueError("Unknown preprocessing strategy")
        if not isinstance(self.min_observations, int) or self.min_observations < 1:
            raise ValueError("min_observations must be a positive integer")
        numeric = self._read(X)
        counts = numeric.notna().groupby(X._platform).sum()
        keep = numeric.notna().sum().ge(self.min_observations) & numeric.nunique().gt(1)
        if self.strategy == "shared_median":
            keep &= counts.ge(self.min_observations).all()
        self.selected_features_ = keep.index[keep].tolist()
        if not self.selected_features_: raise ValueError("No supported, varying behavioral features in training")
        self.dropped_features_ = keep.index[~keep].tolist()
        self.log_features_ = [name for name in self.selected_features_
                              if BASELINE_PROPERTIES[name]["x-linear-transform"] == "log1p"]
        values = self._values(X)
        self.medians_ = values.median()
        self.platform_medians_ = values.groupby(X._platform).median().where(counts[self.selected_features_].ge(self.min_observations))
        if self.strategy != "native":
            filled = self._fill(values, X._platform)
            self.means_ = filled.mean()
            self.scales_ = filled.std(ddof=0).replace(0, 1)
        return self

    def transform(self, X):
        check_is_fitted(self, "selected_features_")
        values = self._values(X)
        mask = values.isna().to_numpy(dtype=float)
        if self.strategy != "native":
            values = (self._fill(values, X._platform)-self.means_)/self.scales_
        result = values.to_numpy(dtype=float)
        return np.concatenate([result, mask], axis=1) if self.add_missing_indicators else result

    def get_feature_names_out(self, input_features=None):
        check_is_fitted(self, "selected_features_")
        names = list(self.selected_features_)
        if self.add_missing_indicators: names += [name + "__missing" for name in names]
        return np.asarray(names, dtype=object)

def make_baseline_model(strategy="shared_median", min_observations=1, allow_pooled_fallback=False):
    """Unfitted pipeline. Call fit only on independently labeled, fold-specific account rows."""
    preprocessing = BaselinePreprocessor(strategy=strategy, min_observations=min_observations,
        allow_pooled_fallback=allow_pooled_fallback, add_missing_indicators=strategy != "native")
    estimator = (HistGradientBoostingClassifier(random_state=42, early_stopping=False)
                 if strategy == "native" else LogisticRegression(max_iter=2000))
    return Pipeline([("preprocess", preprocessing), ("classifier", estimator)])

# No labels are fabricated and no preprocessing is fitted to the full exported corpus.
# First build training/validation account tables from their permitted comment corpora/windows.
# model = make_baseline_model("shared_median")
# model.fit(train_accounts, y_train)
# scores = model.predict_proba(validation_accounts)[:, 1]
# Alternatives: "native", or "platform_median" with allow_pooled_fallback=True only
# after validating that cross-platform medians are a defensible approximation.
print("Baseline model factories ready: shared_median, platform_median, native")


## 8. Export location

The current dataset has exactly the three requested columns. No account-feature or public-copy exports are generated.


In [ ]:
import getpass


import hmac

def pseudonymise(frame, salt):
    """Pseudonymous, not anonymous: behavior can still permit reidentification."""
    if not salt: raise ValueError("empty salt")
    out = frame.copy()
    platform = out["_platform"].fillna("") if "_platform" in out else pd.Series("", index=out.index)
    identity_domains = {"global_user_id": "account", "user_id": "user", "username": "handle",
                        "post_id": "comment", "parent_comment_id": "comment", "thread_id": "comment",
                        "source_post_id": "source"}
    for col, domain in identity_domains.items():
        if col not in out: continue
        values = []
        for plat, value in zip(platform, out[col]):
            if value is None or pd.isna(value) or str(value) == "": values.append(None); continue
            message = json.dumps([domain, str(plat), str(value)], ensure_ascii=False).encode()
            values.append(hmac.new(salt.encode(), message, hashlib.sha256).hexdigest())
        out[col] = values
    return out.drop(columns=["text_content", "clean_text", "display_name", "bio_text", "location",
        "_source_url", "_raw_ref", "_profile_user_id", "hashtags", "user_mentions", "urls", "user_recent_posts",
        "user_recent_timestamps", "created_at", "created_at_dt", "account_created_at", "_collected_at",
        "_profile_collected_at", "_capture_session", "profile_checked_at"], errors="ignore")

# The three-column export contains no account identifiers to pseudonymise.
print("Scraped comment data:", FEATURES / "dataset.csv")
print("Columns: like_count, reply_count, date_published")


## 9. Validation, modeling limits, and migration

See [CODE_REVIEW.md](CODE_REVIEW.md) for the code findings, column-by-column critique,
feature baseline, complexity analysis, and leakage-safe evaluation plan.

- Run `python -m unittest discover -s tests -v` for offline regression tests. They do not
  open browsers or request credentials. Live payload compatibility still needs validation.
- `source_post_id` identifies the source video/post; `post_id` identifies the comment;
  `parent_comment_id` identifies its direct parent. `thread_id` is the top-level comment
  thread except on X, where it is the conversation root. X targets should be root tweets.
- `global_user_id` is platform plus stable account ID. TikTok now prefers `uid`; rebuild
  canonical data from raw and migrate existing labels keyed to `sec_uid` deliberately.
- Browser raw archives accumulate history. Rebuilding canonical files replays that history,
  not only today's target list. Explicitly filter source posts and time windows for modeling.
- Browser caps reset per target. Raw responses can overshoot; replay enforces each new
  capture session's requested cap. Legacy raw records have no recoverable cap metadata.
- Reported totals are hints, not completeness percentages. The browser stops on cap,
  unique-comment plateau, or batch ceiling; a plateau does not establish completeness.
- Profiles include stable identity and snapshot dates. Old/identity-less cache entries
  require refresh; failed refreshes retain usable prior snapshots marked stale. Unknown
  values stay missing. Facebook profile enrichment remains unimplemented.
- `dataset_accounts.csv` contains candidate features, not a validated model. Use the explicit
  `BASELINE_ACCOUNT_FEATURE_COLUMNS` for a first experiment and evaluate profile additions
  separately. `feature_quality.csv` reports missingness and constant columns per platform.
- Full-corpus coordination/repetition features are exploratory: recompute them in each
  permitted training/inference context after defining account/campaign/time splits.
- The saved sample has only one post per platform and mostly single-comment accounts.
  More posts, repeated observations, independent labels, and comparison groups are needed.
- New CSVs use an explicit null marker; read them with `read_csv_with_lists`. New public
  exports hash all identity/locator columns and remove free text and exact timestamps.
  Regenerate comment/account public tables together: their HMAC IDs differ from old exports.
- Keep profile cookies, API credentials, raw content, and pseudonymisation salts private.
  Public exports are pseudonymous, not anonymous. Reopen the notebook after file-level edits
  so an editor's older in-memory copy does not overwrite them.
